# Utils y librerías




In [ ]:
pip install unidecode

In [ ]:
!pip install thefuzz

In [ ]:
!pip install tokenizers

In [ ]:
!pip install transformers

In [ ]:
!pip install pysentimiento

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# UTILS
from unidecode import unidecode
from thefuzz import process
from nltk.tokenize import word_tokenize
from openai import OpenAI
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from bs4 import BeautifulSoup
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer
from nltk.corpus import stopwords
from itertools import chain
from collections import Counter


import pandas as pd
import unicodedata
import glob
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch
import nltk
import re
import string
import textwrap
import ast



In [ ]:
imagenes_eda_dir = (
    "/content/drive/MyDrive/estudio/UNAL/new_project/graphics/imagenes_EDA/"
)
os.makedirs(imagenes_eda_dir, exist_ok=True)

imagenes_grupos_dir = (
    "/content/drive/MyDrive/estudio/UNAL/new_project/graphics/imagenes_grupos/"
)
os.makedirs(imagenes_grupos_dir, exist_ok=True)


In [ ]:
pd.set_option('display.max_rows', None)

# Funciones de análisis de texto

## Generación de bigramas

In [ ]:
def clean_text(text):
    text = text.lower()  # Lowercase
    text = re.sub(r'\d+', '', text)  # Remove numbers
    text = text.translate(str.maketrans('', '', string.punctuation))  # Remove punctuation
    text = re.sub(r'\W', ' ', text)  # Remove special characters
    text = BeautifulSoup(text, "html.parser").get_text()  # Remove HTML tags
    return text

def clean_corpus(col, df):
   #cleaning the given column

  cleaned_df = df.dropna(subset=[col])
  cleaned_df[col] = cleaned_df[col].str.strip().str.lower()
  cleaned_df = cleaned_df.drop_duplicates(subset=[col])
  print("Total responses:", len(cleaned_df))

  print("Cleaning text")
  text = cleaned_df[col].astype(str).to_numpy()
  cc = [clean_text(doc) for doc in text]
  return cc

def get_bigrams(stopwords_es_manual, cleaned_corpus):
  from sklearn.feature_extraction.text import TfidfVectorizer
  import numpy as np

  stopwords_nltk = set(stopwords.words('spanish'))
  todas_las_stopwords = stopwords_nltk.union(set(stopwords_es_manual))

  vectorizer = TfidfVectorizer(
      stop_words=list(todas_las_stopwords),
      ngram_range=(2,4),  # include bigrams
      min_df=2
  )

  X = vectorizer.fit_transform(cleaned_corpus)

  # Average importance of each term
  mean_tfidf = np.asarray(X.mean(axis=0)).flatten()

  terms = vectorizer.get_feature_names_out()
  top_indices = mean_tfidf.argsort()[::-1][:10]

  for i in top_indices:
      print(terms[i], mean_tfidf[i])


def main_get_bigrams(col, df, stopwords_es_manual):
  cc = clean_corpus(col, df)
  get_bigrams(stopwords_es_manual, cc)




## Análisis por temas (topics)

In [ ]:
def get_topics(stopwords_es_manual, cleaned_corpus):

    stopwords_nltk = set(stopwords.words('spanish'))
    todas_las_stopwords = stopwords_nltk.union(set(stopwords_es_manual))

    documents = cleaned_corpus

    count_vect = CountVectorizer(stop_words=list(todas_las_stopwords))
    doc_term_matrix = count_vect.fit_transform(documents)

    feature_names = count_vect.get_feature_names_out()

    n_components = 2 # Number of topics
    lda = LatentDirichletAllocation(n_components=n_components, random_state=42)
    lda.fit(doc_term_matrix)

    for topic_idx, topic in enumerate(lda.components_):
        print(f"Topic #{topic_idx}:")
        top_words_idx = topic.argsort()[:-10 - 1:-1]
        top_words = [feature_names[i] for i in top_words_idx]
        print(f"{', '.join(top_words)}\n")

    doc_topic_dist = lda.transform(doc_term_matrix)
    print("Distribución de topics:")
    print(doc_topic_dist)

def main_get_topics(col, df, stopwords_es_manual):
      cc = clean_corpus(col, df)
      get_topics(stopwords_es_manual, cc)


## Top 15 palabras más importantes

In [ ]:
def limpiar_y_contar(serie):

    nltk.download('stopwords')
    stopwords_nltk = set(stopwords.words('spanish'))
    todas_las_stopwords = stopwords_nltk.union(set(stopwords_es_manual))

    texto_completo = " ".join(serie.dropna().astype(str)).lower()
    palabras = re.findall(r'\b\w+\b', texto_completo)

    palabras_limpias = [p for p in palabras if p not in todas_las_stopwords and len(p) > 2]
    return Counter(palabras_limpias)

def get_top_15(col, df, stopwords_es_manual):
    conteo = limpiar_y_contar(df[col])
    print("Top 15 palabras más importantes:")
    for palabra, total in conteo.most_common(15):
        print(f"{palabra}: {total}")

## SaBERT-Spanish-Sentiment-Analysis

In [ ]:
from transformers import BertForSequenceClassification, BertTokenizer
model = BertForSequenceClassification.from_pretrained("VerificadoProfesional/SaBERT-Spanish-Sentiment-Analysis")
tokenizer = BertTokenizer.from_pretrained("VerificadoProfesional/SaBERT-Spanish-Sentiment-Analysis")


In [ ]:
def predict_sabert(model,tokenizer,text,threshold = 0.5):
        inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128)
        with torch.no_grad():
            outputs = model(**inputs)

        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=1).squeeze().tolist()

        predicted_class = torch.argmax(logits, dim=1).item()
        if probabilities[predicted_class] <= threshold and predicted_class == 1:
            predicted_class = 0

        return bool(predicted_class), probabilities

## Pysentimiento - sentimientos

In [ ]:
from pysentimiento import create_analyzer
sentiment_analyzer = create_analyzer(task="sentiment", lang="es")

## Pysentimient - emocion

In [ ]:
emotion_analyzer = create_analyzer(task="emotion", lang="es")

# Análisis de caracterización

In [ ]:
char_file_path = '/content/drive/MyDrive/estudio/UNAL/new_project/data/generated/standardize/standardized-characterization-file.csv'


In [ ]:
char_file = pd.read_csv(char_file_path)


In [ ]:
len(char_file)

## Detallar EDA, por variable.

In [ ]:
char_file.columns

**username y uncode_usr**

pueden ser diferentes. Incluso hay usuarios que no cuentan con uncode_usr (al parecer porque no necesariamente tenian que estar inscritos a uncode_usr para participar en el curso o llenar la encuesta).

In [ ]:
char_file[['username', 'uncode_usr']]

**email**

no hay valores de NA para el email. Por esto, podria considerarse como un buen identificador de los usuarios en análisis posteriores.

In [ ]:
char_file['email'].isna().sum()

También se verifica que no haya cadenas de caracteres vacíos. **No hay.**

In [ ]:
char_file[char_file['email'].str.len() < 16]

**Ocupación**

Se realiza el conteo de las ocupaciones

In [ ]:
plt.figure(figsize=(10,6))
sns.countplot(data=char_file, y='ocupacion', order=char_file['ocupacion'].value_counts().index)

plt.xlabel("Cantidad")
plt.ylabel("Ocupación")
nombre_archivo = os.path.join(imagenes_eda_dir, "1_grafica_ocupaciones.pdf")
plt.savefig(nombre_archivo, format="pdf", bbox_inches="tight", pad_inches=0.1)

plt.show()

In [ ]:
char_file["ocupacion"].value_counts(dropna=False)

La mayoría de personas inscritas son estudiantes. Para esta columna no existen NA.

**otra ocupacion**

se muestra gráficamente cómo se ve la variable otra ocupación

In [ ]:
def wrap_labels(ax, width):
    labels = []
    for label in ax.get_yticklabels():
        wrapped_text = textwrap.fill(label.get_text(), width)
        labels.append(wrapped_text)
    ax.set_yticklabels(labels)

plt.figure(figsize=(10,8))
ax = sns.countplot(
    data=char_file,
    y='otra_ocupacion',
    order=char_file['otra_ocupacion'].value_counts().index
)

wrap_labels(ax, 20)

plt.xlabel("Cantidad")
plt.ylabel("Otra ocupación")
##nombre_archivo = os.path.join(imagenes_eda_dir, "grafica_otra_ocupacion.pdf")
##plt.savefig(nombre_archivo, format="pdf", bbox_inches="tight", pad_inches=0.1)

plt.tight_layout()
plt.show()

In [ ]:


char_file_filtrado = char_file[char_file['otra_ocupacion'] != 'campo_corregido']
char_file_filtrado = char_file_filtrado[char_file_filtrado['otra_ocupacion'] != 'asasdadda']

plt.figure(figsize=(10,8))

ax = sns.countplot(
    data=char_file_filtrado,
    y='otra_ocupacion',
    order=char_file_filtrado['otra_ocupacion'].value_counts().index
)

wrap_labels(ax, 20)

plt.xlabel("Cantidad")
plt.ylabel("Otra ocupación")
nombre_archivo = os.path.join(imagenes_eda_dir, "2_grafica_otra_ocupacion.pdf")
plt.savefig(nombre_archivo, format="pdf", bbox_inches="tight", pad_inches=0.1)
plt.tight_layout()
plt.show()

In [ ]:
char_file["otra_ocupacion"].value_counts(dropna=False)

**dedicación semanal**

In [ ]:
orden_deseado = ['1-2 horas', '3-4 horas', '5-6 horas', '7 horas o más']

plt.figure(figsize=(10,6))

sns.countplot(
    data=char_file,
    y='dedicacionsemanal',
    order=orden_deseado
)

plt.xlabel("Cantidad")
plt.ylabel("Dedicación semanal")
nombre_archivo = os.path.join(imagenes_eda_dir, "3_grafica_dedicacion_semanal.pdf")
plt.savefig(nombre_archivo, format="pdf", bbox_inches="tight", pad_inches=0.1)
plt.show()

**nivel de programación**

In [ ]:

orden_deseado = ['nulo', 'basico', 'intermedio', 'avanzado']

plt.figure(figsize=(10,6))

sns.countplot(
    data=char_file,
    y='nivelprogramacion',
    order=orden_deseado
)

##plt.title('Nivel de programación')
plt.xlabel("Cantidad")
plt.ylabel("Nivel de programación")
nombre_archivo = os.path.join(imagenes_eda_dir, "4_grafica_nivel_programacion.pdf")
plt.savefig(nombre_archivo, format="pdf", bbox_inches="tight", pad_inches=0.1)

plt.show()

**lenguajes de programación y otro_lenguaje**

A continuación se puede ver la cantidad de personas que indicaron uno, dos, tres o cuatro lenguajes de programación

In [ ]:
plt.figure(figsize=(10, 6))

ax = sns.countplot(data=char_file, y='conteo_lenguajes', palette='viridis')

#plt.title('Distribución de cantidad de lenguajes por persona')
plt.ylabel('Número de lenguajes conocidos')
plt.xlabel('Cantidad de personas')

for p in ax.patches:
    width = p.get_width()
    ax.annotate(f'{int(width)}',
                (width, p.get_y() + p.get_height() / 2.),
                ha = 'left', va = 'center',
                xytext = (5, 0),
                textcoords = 'offset points')

plt.xlim(0, char_file['conteo_lenguajes'].value_counts().max() * 1.1)
##nombre_archivo = os.path.join(imagenes_eda_dir, "grafica_lenguajes_programacion.pdf")
##plt.savefig(nombre_archivo, format="pdf", bbox_inches="tight", pad_inches=0.1)
plt.show()

Se transforma la lista de lenguajes a una lista de python

Ahora, se observa la distribución general de lenguajes de programación:

In [ ]:
char_file['lenguajes_programacion'] = char_file['lenguajes_programacion'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

In [ ]:
lenguajes_unicos = char_file['lenguajes_programacion'].explode()
conteo_lenguajes = lenguajes_unicos.value_counts()

df_conteo = conteo_lenguajes.reset_index()
df_conteo.columns = ['Lenguaje', 'Frecuencia']

mapeo_nombres = {
    'c_c_plus': 'C++',
    'c_sharp': 'C#',
    'python': 'Python',
    'javascript': 'JavaScript',
    'java': 'Java',
    'r': 'R',
    'otro': 'Otro',

    'no_programa': 'No programa'
}

df_conteo['Lenguaje'] = df_conteo['Lenguaje'].replace(mapeo_nombres)

plt.figure(figsize=(10, 8))
sns.barplot(data=df_conteo, y='Lenguaje', x='Frecuencia', palette='magma')

plt.xlabel('Número de Usuarios')
plt.ylabel("Lenguajes de programación")

nombre_archivo = os.path.join(imagenes_eda_dir, "5_grafica_lenguajes_programacion.pdf")
plt.savefig(nombre_archivo, format="pdf", bbox_inches="tight", pad_inches=0.1)

plt.show()

Como en la variable de otro_lenguaje se logró recuperar información valiosa, hacemos un conteo de qué palabras se repiten más en dicho campo, para ver qué otros lenguajes conocen los participantes del curso

Al obtener un top 15 de las palabras más importantes, usando un diccionario de stopwords personalizado, se ve que el más repetido aún es python pero también visual basic, sql, html, php

In [ ]:
char_file['otro_lenguaje'].value_counts(dropna=False)

In [ ]:
stopwords_es_manual = [
    'ninguno', 'ninguna', 'aplica', 'solo', 'programación', 'usado', 'programado', 'nunca',
    'es_spam_ruido', 'declaro_sin_experiencia', 'campo_corregido'
]
get_top_15("otro_lenguaje", char_file, stopwords_es_manual)

También se hace un análisis de bigramas

In [ ]:
stopwords_es_manual = [
    'ninguno', 'ninguna', 'aplica', 'solo', 'programación', 'nunca', 'programado',
    'primera', 'vez', 'algun', 'lenguaje', 'usado', 'es_spam_ruido', 'declaro_sin_experiencia',
    'campo_corregido'
]
main_get_bigrams("otro_lenguaje", char_file, stopwords_es_manual)

y se generan topics

In [ ]:
stopwords_es_manual = [
    'ninguno', 'ninguna', 'aplica', 'solo', 'programación', 'nunca', 'programado',
    'primera', 'vez', 'algun', 'lenguaje', 'apenas', 'basico', 'hace', 'uso', 'na',
    'aprender', 'programar', 'ningún', 'conocimientos', 'es_spam_ruido',
    'declaro_sin_experiencia', 'campo_corregido'
]
main_get_topics("otro_lenguaje", char_file, stopwords_es_manual)

**dónde conoció el curso y otro_medio**

In [ ]:
plt.figure(figsize=(10,6))
sns.countplot(data=char_file, y='dondeconocioelcurso', order=char_file['dondeconocioelcurso'].value_counts().index)

#plt.title('Dónde conoció el curso')
plt.xlabel("Cantidad")
plt.ylabel("Dónde conoció el curso")
nombre_archivo = os.path.join(imagenes_eda_dir, "6_grafica_donde_conocio.pdf")
plt.savefig(nombre_archivo, format="pdf", bbox_inches="tight", pad_inches=0.1)
plt.show()

Se hace un análisis con el texto que queda en otro_medio_clean.

El otro medio más común fueron profesores.

In [ ]:
stopwords_es_manual = [

    'ninguno', 'ninguna', 'aplica', 'curso',
    'programacion', 'colombia', 'programación',
    'computadores', 'campo_corregido', 'declaro_sin_experiencia'
]
get_top_15("otro_medio", char_file, stopwords_es_manual)

profesor de universidad, universidad nacional, medio profesor o alguna materia

In [ ]:
stopwords_es_manual = [

    'ninguno', 'ninguna', 'aplica', 'curso',
    'programacion', 'colombia', 'programación'
]
main_get_bigrams("otro_medio", char_file, stopwords_es_manual)

In [ ]:
stopwords_es_manual = [

    'ninguno', 'ninguna', 'aplica', 'curso',
    'programacion', 'colombia', 'programación'
]
main_get_topics("otro_medio", char_file, stopwords_es_manual)

**motivación y otra motivación**

In [ ]:
plt.figure(figsize=(10,8))
ax = sns.countplot(
    data=char_file,
    y='motivacion',
    order=char_file['motivacion'].value_counts().index
)

wrap_labels(ax, 40)

#plt.title('Motivación')
plt.ylabel('Motivación')
plt.xlabel("Cantidad")
nombre_archivo = os.path.join(imagenes_eda_dir, "7_grafica_motivacion.pdf")
plt.savefig(nombre_archivo, format="pdf", bbox_inches="tight", pad_inches=0.1)
plt.tight_layout()
plt.show()

Conteo de palabras más repetidas:

In [ ]:
stopwords_es_manual = [
    'ninguno', 'ninguna', 'general', 'declaro_sin_experiencia'
]

get_top_15("otra_motivacion", char_file, stopwords_es_manual)

Se procede a hacer un análisis de bigramas más repetidos.



In [ ]:
stopwords_es_manual = [
    'quiero', 'aprender', 'programacion', 'programación', 'deseo',
    'tener', 'poder', 'programar', 'mundo',
    'realizar', 'interesa', 'mas'
]

main_get_bigrams("otra_motivacion", char_file, stopwords_es_manual)

análisis de topics

In [ ]:

stopwords_es_manual = [
    'quiero', 'aprender', 'programacion', 'programación', 'deseo',
    'tener', 'poder', 'programar', 'mundo',
    'realizar', 'interesa', 'mas'
]

cc = clean_corpus("otra_motivacion", char_file)
get_topics(stopwords_es_manual, cc)

## Analizar la razón por la que se inscribieron al curso (campo de texto abierto).

Análisis de bigramas:

In [ ]:
stopwords_es_manual = [
    'quiero', 'aprender', 'programacion', 'programación', 'deseo',
    'tener', 'poder', 'programar', 'mundo',
    'realizar', 'interesa', 'mas', 'nacional', 'colombia'
]

main_get_bigrams("razon_realizar_curso", char_file, stopwords_es_manual)

Análisis de texto por medio de topics

In [ ]:

stopwords_es_manual = [
    'quiero', 'aprender', 'programacion', 'programación', 'deseo',
    'tener', 'poder', 'programar', 'mundo',
    'realizar', 'interesa', 'mas'
]

cc = clean_corpus("razon_realizar_curso", char_file)
get_topics(stopwords_es_manual, cc)



In [ ]:
stopwords_es_manual = [
    'quiero', 'aprender', 'programacion', 'programación', 'deseo',
    'tener', 'poder', 'programar', 'mundo',
    'realizar', 'interesa', 'mas'
]

get_top_15("razon_realizar_curso", char_file, stopwords_es_manual)

# Análisis de autoevaluación


In [ ]:
eval_file_path = '/content/drive/MyDrive/estudio/UNAL/new_project/data/generated/standardize/standardized-evaluation-file.csv'


In [ ]:
eval_file = pd.read_csv(eval_file_path)


In [ ]:
eval_file.columns

In [ ]:
len(eval_file)

## Estadísticas generales del curso a partir de las preguntas cerradas.


In [ ]:
questions_dict_type_1 = {
    "pregunta1" : "Mejora en programacion",
    "pregunta2" : "Notebooks de Python",
    "pregunta3" : "Pruebas de código",
    "pregunta4" : "Retroalimentación",
    "pregunta11" : "Uso de pistas",
    "pregunta12" : "Utilidad uncode",
    "pregunta13" : "Videos",
    "pregunta14" : "Visualización de código",
    "pregunta18" : "Foro",
}

In [ ]:
questions_dict_type_2 = {
    "pregunta5" : "Abstraer un problema en forma de algoritmo e implementarlo en un programa con el lenguaje de programación Python.",
    "pregunta6" : "Conocer y aplicar los principios de manejo de variables, tipos de datos y operaciones entre valores numéricos y de texto en Python.",
    "pregunta7" : "Diseñar e implementar algoritmos con decisiones basadas en la evaluación de expresiones booleanas.",
    "pregunta8" : "Diseñar e implementar algoritmos con la aplicación de ciclos iterativos.",
    "pregunta9" : "Distinguir y elegir la colección apropiada para la resolución de un problema con una estructura de datos compleja.",
    "pregunta10" : "Usar colecciones en el diseño e implementación de algoritmos para la solución de problemas.",
    "pregunta17" : "Declarar y manipular listas, tuplas, matrices, diccionarios y conjuntos con el lenguaje de programación Python.",
}

In [ ]:
orden_deseado_type_1 = ['1-totalmente en desacuerdo',
                 '2-en desacuerdo',
                 '3-algo en desacuerdo',
                 '4-algo de acuerdo',
                 '5-de acuerdo',
                 '6-totalmente de acuerdo',
                 '0-no aplica (no conozco esta funcionalidad)']

In [ ]:
orden_deseado_type_2 = ['1-muy bajo',
                 '2-bajo',
                 '3-medio',
                 '4-alto',
                 '5-muy alto']

In [ ]:
def get_results_types(df, dict, order):
  results = {}
  for q_id in dict.keys():
      if q_id in df.columns:
          counts = df[q_id].str.strip().value_counts()
          ordered_series = counts.reindex(order, fill_value=0)
          nombre_pregunta = dict[q_id]
          results[nombre_pregunta] = ordered_series.astype(int).tolist()

  return(results)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import textwrap

def survey_latex(results, category_names):

    labels = [textwrap.fill(l, width=30) for l in results.keys()]
    data = np.array(list(results.values()))

    data_perc = (data.T / data.sum(axis=1)).T * 100
    data_cum = data_perc.cumsum(axis=1)

    category_colors = [
        '#8b0000', '#f06292', '#e57373',
        '#a5d6a7', '#4caf50', '#1b5e20',
        '#9e9e9e'
    ]

    fig, ax = plt.subplots(figsize=(10, 5), dpi=300)
    ax.invert_yaxis()

    for i, (colname, color) in enumerate(zip(category_names, category_colors)):
        widths = data_perc[:, i]
        starts = data_cum[:, i] - widths

        ax.barh(labels, widths, left=starts, height=0.6,
                label=colname, color=color, edgecolor='white', linewidth=0.5)

        xcenters = starts + widths / 2

        text_color = 'white' if color in ['#8b0000', '#1b5e20', '#9e9e9e'] else 'black'

        for y, (x, perc, count) in enumerate(zip(xcenters, widths, data[:, i])):
            if perc > 3.5:
                ax.text(x, y, str(int(count)), ha='center', va='center',
                        color=text_color, fontsize=8, fontweight='bold')

    ax.set_xlim(0, 100.5)
    ax.margins(x=0)
    ax.set_xticks(np.arange(0, 101, 10))
    ax.set_xticklabels([f'{i}%' for i in range(0, 101, 10)], fontsize=9)
    ax.tick_params(axis='y', labelsize=9)

    ax.grid(axis='x', color='gray', alpha=0.3, linestyle='--', linewidth=0.7)
    ax.set_axisbelow(True)

    legend = ax.legend(
        ncol=len(category_names) // 2 + 1,
        bbox_to_anchor=(0.5, -0.15),
        loc='upper center',
        fontsize=8,
        frameon=True,
        fancybox=True
    )

    plt.tight_layout()

    return fig, ax

fig, ax = survey_latex(get_results_types(eval_file, questions_dict_type_1, orden_deseado_type_1), orden_deseado_type_1)


nombre_archivo = os.path.join(imagenes_eda_dir, "8_grafica_likert_1.pdf")
plt.savefig(nombre_archivo, format="pdf", bbox_inches="tight", pad_inches=0.1)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import textwrap

def survey_latex_long_text(results, category_names):

    labels = [textwrap.fill(l, width=48) for l in results.keys()]
    data = np.array(list(results.values()))

    data_perc = (data.T / data.sum(axis=1)).T * 100
    data_cum = data_perc.cumsum(axis=1)

    category_colors = [
        '#8b0000', '#f06292', '#e57373',
        '#a5d6a7', '#4caf50'
    ]

    fig, ax = plt.subplots(figsize=(11, 7.5), dpi=300)
    ax.invert_yaxis()

    for i, (colname, color) in enumerate(zip(category_names, category_colors)):
        widths = data_perc[:, i]
        starts = data_cum[:, i] - widths

        ax.barh(labels, widths, left=starts, height=0.55,
                label=colname, color=color, edgecolor='white', linewidth=0.5)

        xcenters = starts + widths / 2

        text_color = 'white' if color in ['#8b0000', '#4caf50'] else 'black'

        for y, (x, perc, count) in enumerate(zip(xcenters, widths, data[:, i])):
            if perc > 3.5:
                ax.text(x, y, str(int(count)), ha='center', va='center',
                        color=text_color, fontsize=8.5, fontweight='bold')

    ax.set_xlim(0, 100.5)
    ax.margins(x=0)
    ax.set_xticks(np.arange(0, 101, 10))
    ax.set_xticklabels([f'{i}%' for i in range(0, 101, 10)], fontsize=9)

    ax.tick_params(axis='y', labelsize=8.5, pad=6)

    ax.grid(axis='x', color='gray', alpha=0.3, linestyle='--', linewidth=0.7)
    ax.set_axisbelow(True)

    ax.legend(
        ncol=len(category_names),
        bbox_to_anchor=(0.5, -0.12),
        loc='upper center',
        fontsize=8.5,
        frameon=True,
        fancybox=True
    )

    plt.tight_layout()
    return fig, ax

fig, ax = survey_latex_long_text(
    get_results_types(eval_file, questions_dict_type_2, orden_deseado_type_2),
    orden_deseado_type_2
)


nombre_archivo = os.path.join(imagenes_eda_dir, "9_grafica_likert_2.pdf")
plt.savefig(nombre_archivo, format="pdf", bbox_inches="tight", pad_inches=0.1)

plt.show()

## Análisis temático o de contenido sobre lo positivo y lo negativo en la autoevaluación, para saber la percepción general del curso. Análisis de las preguntas abiertas (pregunta15, pregunta16).

### Pregunta 15: ¿Cuál considera que es la mayor fortaleza de este módulo que le ayudó a alcanzar los objetivos de aprendizaje?

In [ ]:
stopwords_es_manual = [
    'mayor', 'fortaleza', 'cada', 'tema', 'gran', 'ayuda',
    'proceso', 'aprendizaje', 'bien', 'curso', 'forma',
    'modulo', 'manera'
]
get_top_15("pregunta15", eval_file, stopwords_es_manual)

In [ ]:
stopwords_es_manual = [
    'mayor', 'fortaleza', 'cada', 'tema', 'gran', 'ayuda',
    'proceso', 'aprendizaje', 'bien', 'curso', 'forma',
    'modulo', 'manera'
]
main_get_bigrams("pregunta15", eval_file, stopwords_es_manual)

In [ ]:
stopwords_es_manual = [
    'mayor', 'fortaleza', 'cada', 'tema', 'gran', 'ayuda',
    'proceso', 'aprendizaje', 'bien', 'curso', 'forma',
    'modulo', 'manera'
]
main_get_topics("pregunta15", eval_file, stopwords_es_manual)

#### SaBERT-Spanish-Sentiment-Analysis

Se hace un análisis de percepción negativa o positiva sobre la pregunta 15. El modelo comprueba que las sensaciones son positivas para esta pregunta sobre las fortalezas.

In [ ]:
raw_results = eval_file['pregunta15'].apply(
    lambda x: predict_sabert(model, tokenizer, str(x), threshold=0.5)
).tolist()

In [ ]:
df_analisis = pd.DataFrame(raw_results, columns=['es_positivo', 'probabilidades'])

In [ ]:
df_analisis['confianza'] = df_analisis['probabilidades'].apply(lambda x: max(x))
df_analisis = df_analisis[['es_positivo', 'confianza', 'probabilidades']]

In [ ]:
df_analisis['texto_original'] = eval_file['pregunta15'].values

In [ ]:
df_analisis[(df_analisis['es_positivo'] == False) &  (df_analisis['confianza'] < 0.6)][['texto_original', 'confianza']]

In [ ]:
df_analisis['es_positivo'].value_counts()

#### pysentimiento - sentimientos

In [ ]:
py_sentimento_raw_results = eval_file['pregunta15'].apply(
    lambda x: sentiment_analyzer.predict(str(x))
).tolist()

In [ ]:
py_sentimento_raw_results

In [ ]:
data_list = []

for item in py_sentimento_raw_results:
    fila = {
        'sentimiento_final': item.output,
        **item.probas
    }
    data_list.append(fila)

df_resultados_pregunta15_py = pd.DataFrame(data_list)

In [ ]:
print(df_resultados_pregunta15_py['sentimiento_final'].value_counts())

In [ ]:
df_final_resultados_pregunta15_py = pd.concat([eval_file['pregunta15'].reset_index(drop=True), df_resultados_pregunta15_py], axis=1)

In [ ]:
df_final_resultados_pregunta15_py

#### pysentimiento - emoción

In [ ]:
py_sentimento_emotion_raw_results = eval_file['pregunta15'].apply(
    lambda x: emotion_analyzer.predict(str(x))
).tolist()

In [ ]:
data_list = []

for item in py_sentimento_emotion_raw_results:
    fila = {
        'emocion': item.output,
        **item.probas
    }
    data_list.append(fila)

df_resultados_pregunta15_py_emocion = pd.DataFrame(data_list)

In [ ]:
print(df_resultados_pregunta15_py_emocion['emocion'].value_counts())

### Pregunta 16: ¿Qué aspectos considera que podemos mejorar en este curso para que le permita alcanzar en mayor medida los objetivos de aprendizaje?

In [ ]:
stopwords_es_manual = [
    'mayor', 'fortaleza', 'cada', 'tema', 'gran', 'ayuda',
    'proceso', 'aprendizaje', 'bien', 'curso', 'forma',
    'modulo', 'manera', "tal", "vez", "muchas", "gracias",
    "puede", "mejorar", "ser", "considero", "creo", "parte"
]

In [ ]:
get_top_15("pregunta16", eval_file, stopwords_es_manual)

In [ ]:
main_get_bigrams("pregunta16", eval_file, stopwords_es_manual)

In [ ]:
main_get_topics("pregunta16", eval_file, stopwords_es_manual)

#### SaBERT-Spanish-Sentiment-Analysis

In [ ]:
raw_results = eval_file['pregunta16'].apply(
    lambda x: predict_sabert(model, tokenizer, str(x), threshold=0.5)
).tolist()

In [ ]:
df_analisis_p16 = pd.DataFrame(raw_results, columns=['es_positivo', 'probabilidades'])

In [ ]:
df_analisis_p16['confianza'] = df_analisis_p16['probabilidades'].apply(lambda x: max(x))
df_analisis_p16 = df_analisis_p16[['es_positivo', 'confianza', 'probabilidades']]

In [ ]:
df_analisis_p16['es_positivo'].value_counts()

Puede que haya cosas positivas en la columna de cosas por mejorar.

#### pysentimiento - sentimientos

In [ ]:
py_sentimento_raw_results = eval_file['pregunta16'].apply(
    lambda x: sentiment_analyzer.predict(str(x))
).tolist()

In [ ]:
data_list = []

for item in py_sentimento_raw_results:
    fila = {
        'sentimiento_final': item.output,
        **item.probas
    }
    data_list.append(fila)

df_resultados_pregunta16_py = pd.DataFrame(data_list)

In [ ]:
df_resultados_pregunta16_py['sentimiento_final'].value_counts()

#### pysentimiento - emociones

In [ ]:
py_sentimento_emotion_raw_results = eval_file['pregunta16'].apply(
    lambda x: emotion_analyzer.predict(str(x))
).tolist()

In [ ]:
data_list = []

for item in py_sentimento_emotion_raw_results:
    fila = {
        'emocion': item.output,
        **item.probas
    }
    data_list.append(fila)

df_resultados_pregunta16_py_emocion = pd.DataFrame(data_list)

In [ ]:
df_final_resultados_pregunta16_py = pd.concat([eval_file['pregunta16'].reset_index(drop=True), df_resultados_pregunta16_py_emocion], axis=1)

In [ ]:
df_final_resultados_pregunta16_py['emocion'].value_counts()

In [ ]:
df_final_resultados_pregunta16_py[df_final_resultados_pregunta16_py['emocion'] == 'sadness']

In [ ]:
df_final_resultados_pregunta16_py[df_final_resultados_pregunta16_py['emocion'] == 'anger']

# Análisis de Submissions

In [ ]:
sub_file_path = '/content/drive/MyDrive/estudio/UNAL/new_project/data/generated/standardize/standardized-submissions-file.csv'


In [ ]:
sub_file = pd.read_csv(sub_file_path)


In [ ]:
len(sub_file)

In [ ]:
sub_file.columns

**submitted_on - Fecha de entrega**

Se observa en primera medida si hay datos para todos los meses

In [ ]:
from matplotlib.ticker import MaxNLocator  

plt.figure(figsize=(10, 4))
ax = sns.stripplot(data=sub_file, x='mes', y='ano', jitter=True, alpha=0.5, size=3)

ax.yaxis.set_major_locator(MaxNLocator(integer=True))

plt.xlabel("Mes")
plt.ylabel("Año")

nombre_archivo = os.path.join(imagenes_eda_dir, "10_grafica_rango_interacciones.pdf")
plt.savefig(nombre_archivo, format="pdf", bbox_inches="tight", pad_inches=0.1)

plt.show()

In [ ]:
sub_file['solo_fecha'] = pd.to_datetime(sub_file['solo_fecha'])

In [ ]:
conteo_mensual = sub_file['solo_fecha'].dt.to_period('M').value_counts().sort_index()

print("Registros por mes:")
print(conteo_mensual)

Se revisa visualmente cómo es la distribución de submissions por mes

In [ ]:

resumen = sub_file.set_index('solo_fecha').resample('MS').size()

plt.figure(figsize=(12, 6))
resumen.plot(kind='bar', color='teal', edgecolor='black')

#plt.title('Distribución de Registros por Mes', fontsize=14)
plt.xlabel('Periodo', fontsize=12)
plt.ylabel('Cantidad', fontsize=12)

plt.xticks(
    ticks=range(len(resumen)),
    labels=[d.strftime('%Y-%m') for d in resumen.index],
    rotation=45
)

plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
nombre_archivo = os.path.join(imagenes_eda_dir, "11_grafica_registros_mes.pdf")
plt.savefig(nombre_archivo, format="pdf", bbox_inches="tight", pad_inches=0.1)
plt.show()

Se realiza sobre los usuarios del submission, no necesariamente tienen que tener caracterización o autoevaluación.

Se crea un df para agrupar los usuarios por plataforma

In [ ]:
submissions_user_counts = sub_file.groupby(['username', 'platform']).size().reset_index(name='count')

In [ ]:
len(submissions_user_counts)

In [ ]:
num_duplicados = submissions_user_counts.duplicated(subset=['username', 'platform']).sum()

if num_duplicados > 0:
    print(f" Hay {num_duplicados} registros duplicados.")
else:
    print(" No hay usuarios duplicados por esa combinación.")

A los usuarios por plataforma se les trae su información, pero nos quedamos solo con la fecha limpiada previamente y se trae la fecha de del primer submissions y del último entregado.


In [ ]:
usuarios_y_fechas = pd.merge(
        submissions_user_counts,
        sub_file,
        how='left',
        left_on=['username', 'platform'], right_on=['username', 'platform'], indicator=True)

In [ ]:
usuarios_y_fechas.columns

In [ ]:
usuarios_y_fechas.head()

In [ ]:
len(usuarios_y_fechas)

Se calculan los dias totales basados en dichas fechas de entrega.

In [ ]:
rango_actividad = usuarios_y_fechas.groupby(['username', 'platform'])['solo_fecha'].agg(['min', 'max']).reset_index()
rango_actividad.columns = ['username', 'platform', 'fecha_inicio', 'fecha_final']
rango_actividad['dias_totales'] = (rango_actividad['fecha_final'] - rango_actividad['fecha_inicio']).dt.days


In [ ]:
rango_actividad.head()

### Submissions por usuario (sin actividad) con el periodo de tiempo que estuvo - submissiosn realizados en ese periodo de tiempo.

In [ ]:
submissions_x_usuario = sub_file.groupby(['username', 'platform']).agg(
    submission_x_user=('best', 'size'),
).reset_index()

In [ ]:
submissions_x_usuario.head(10)

In [ ]:
len(submissions_x_usuario)

In [ ]:
usuarios_submissions_fechas = pd.merge(
        submissions_x_usuario,
        rango_actividad,
        how='inner',
        left_on=['username', 'platform'], right_on=['username', 'platform'], indicator=True)

In [ ]:
len(usuarios_submissions_fechas)

In [ ]:
usuarios_submissions_fechas.head(5)

In [ ]:
print("Análisis de Persistencia y Esfuerzo (Total Cohorte):")
stats = usuarios_submissions_fechas[['dias_totales', 'submission_x_user']].describe(percentiles=[.25, .5, .75, .9, .95])
print(stats)

usuarios_submissions_fechas['intensidad'] = usuarios_submissions_fechas['submission_x_user'] / usuarios_submissions_fechas['dias_totales']
print(f"\nIntensidad promedio de la cohorte: {usuarios_submissions_fechas['intensidad'].mean():.2f} intentos/día")

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
limit_dias = usuarios_submissions_fechas['dias_totales'].quantile(0.95)
limit_intentos = usuarios_submissions_fechas['submission_x_user'].quantile(0.95)

sns.scatterplot(data=usuarios_submissions_fechas, x='dias_totales', y='submission_x_user',
                alpha=0.4, color='teal', edgecolor=None)

plt.xlim(0, limit_dias)
plt.ylim(0, limit_intentos)
#plt.title('Mapa de Esfuerzo: Días vs. Intentos (95% de la población)')
plt.xlabel('Días activos en la plataforma')
plt.ylabel('Intentos totales realizados')
plt.grid(True, linestyle='--', alpha=0.5)
nombre_archivo = os.path.join(imagenes_eda_dir, "12_grafica_mapa_esfuerzo.pdf")
plt.savefig(nombre_archivo, format="pdf", bbox_inches="tight", pad_inches=0.1)
plt.show()

El 25% de usuarios tiene dias_totales = 0 y el 50% tiene apenas 9 días.

 Una cuarta parte de la cohorte interactúa con el sistema y se retira (o termina) el mismo día que entra. Esto es común en plataformas abiertas donde muchos usuarios "curiosean" el contenido pero no se quedan.

El curso tiene una fricción inicial alta o un filtro muy rápido. Mucha gente entra, pero la mitad se va antes de cumplir los 10 días.

El máximo de días es 736 y el máximo de intentos es 548. Estos valores están a más de 10 desviaciones estándar de la media.

Solo el 5% de los usuarios supera los 189 días o los 102 intentos.

Basados en estos datos estadísticos, se pueden dividir a los estudiantes en tres grupos: estudiantes de corto, mediano y largo plazo.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import data_table

def segmentar_usuario(row):
    if row['dias_totales'] <= 9:
        return '1. Estudiante de Corto Plazo'
    elif row['dias_totales'] <= 109:
        return '2. Estudiante de Medio Plazo'
    else:
        return '3. Estudiante de Largo Plazo'


usuarios_submissions_fechas['perfil_permanencia'] = usuarios_submissions_fechas.apply(segmentar_usuario, axis=1)
usuarios_submissions_fechas['intensidad_diaria'] = usuarios_submissions_fechas['submission_x_user'] / (usuarios_submissions_fechas['dias_totales'] + 1) # +1 para evitar división por cero

tabla_resumen = usuarios_submissions_fechas.groupby('perfil_permanencia').agg({
    'submission_x_user': ['count', 'mean', 'median'],
    'dias_totales': ['mean', 'median'],
    'intensidad_diaria': 'mean'
}).reset_index()

tabla_resumen.columns = [
    'Perfil de Usuario', 'Total Usuarios', 'Promedio Intentos',
    'Mediana Intentos', 'Promedio Días', 'Mediana Días', 'Intensidad (Intentos/Día)'
]

print("TABLA DE SEGMENTACIÓN DE LA COHORTE (5,938 Usuarios)")
data_table.enable_dataframe_formatter()
display(tabla_resumen)

plt.figure(figsize=(12, 6))
sns.histplot(data=usuarios_submissions_fechas[usuarios_submissions_fechas['dias_totales'] <= 150], x='dias_totales',
             bins=30, kde=True, color='#21918c', alpha=0.7)

plt.axvline(9, color='red', linestyle='--', label='Cierre Exploradores (9 días)')
plt.axvline(109, color='orange', linestyle='--', label='Cierre Persistentes (109 días)')

#plt.title('Embudo de Retención: Distribución de Permanencia en el Curso', fontsize=15)
plt.xlabel('Días de permanencia (Rango 0-150)')
plt.ylabel('Cantidad de usuarios')
plt.legend()
plt.grid(axis='y', alpha=0.3)

nombre_archivo = os.path.join(imagenes_eda_dir, "13_grafica_embudo_retencion.pdf")
plt.savefig(nombre_archivo, format="pdf", bbox_inches="tight", pad_inches=0.1)
plt.show()

## Ratio de intentos para todos los submission

cuántas entregas totales hicieron los usuarios para lograr su mejor nota.

Se realizará sobre el total de submissions, no solamente sobre los que aprobaron y reprobaron el curso.

se puede ver quiénes tuvieron que hacer más intentos para una tarea

In [ ]:
tareas_actividad_usuario = sub_file.groupby(['username', 'platform', 'actividad']).agg(
    submission_activity_user=('best', 'size'),
    max_grade=('grade', 'max')                  # el valor mas alto de esas entregas
).reset_index()

In [ ]:
tareas_actividad_usuario['ratio_intentos'] = tareas_actividad_usuario['submission_activity_user']

ranking_esfuerzo = tareas_actividad_usuario.sort_values(by='ratio_intentos', ascending=False)

print(tareas_actividad_usuario[['username', 'actividad', 'ratio_intentos']].head(10))

Filtramos por un solo usuario para verificar la información

In [ ]:
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'font.sans-serif': 'DejaVu Sans', 'font.size': 11})

plt.figure(figsize=(9, 5))

datos_grafica = tareas_actividad_usuario[tareas_actividad_usuario['ratio_intentos'] <= 25]['ratio_intentos']

ax = sns.histplot(
    datos_grafica,
    discrete=True,       
    color='#2b5c8f',     
    edgecolor='black',
    alpha=0.7
)

plt.xlabel('Ratio de intentos (Total de envíos / Envíos correctos)', fontsize=11, fontweight='bold')
plt.ylabel('Frecuencia (Usuarios × Actividad)', fontsize=11, fontweight='bold')

mediana = tareas_actividad_usuario['ratio_intentos'].median()
plt.axvline(mediana, color='#d95f02', linestyle='--', linewidth=2, label=f'Mediana ({mediana:.1f})')

plt.legend(frameon=True, facecolor='white', edgecolor='none')
plt.tight_layout()

nombre_archivo = os.path.join(imagenes_eda_dir, "14_grafica_distribucion_intentos.pdf")
plt.savefig(nombre_archivo, format="pdf", bbox_inches="tight", pad_inches=0.05)

plt.show()

Se puede ver que la mayoría de usuarios tienen entre 0 y 25 intentos por tarea.

## Actividades con el promedio de intentos más alto para todos los submissions

In [ ]:
dificultad_actividades = (
    tareas_actividad_usuario.groupby('actividad')['ratio_intentos']
    .mean()
    .reset_index()
    .sort_values(by='ratio_intentos', ascending=False)
)

print("Top 10 Actividades con mayor ratio de intentos (más difíciles):")
print(dificultad_actividades.head(10))

La linea de la gráfica indica el ideal (un intento)

In [ ]:
plt.figure(figsize=(12, 8))

sns.barplot(
    data=dificultad_actividades.head(15),
    y='actividad',
    x='ratio_intentos',
    palette='flare'
)

#plt.title('Top 15 Actividades más difíciles (Mayor promedio de intentos)')
plt.xlabel('Promedio de intentos para lograr el éxito')
plt.ylabel('Nombre de la actividad')

plt.axvline(1.0, color='blue', linestyle='--', label='Ideal (1 intento)')
plt.legend()

nombre_archivo = os.path.join(imagenes_eda_dir, "15_grafica_tarea_dificil.pdf")
plt.savefig(nombre_archivo, format="pdf", bbox_inches="tight", pad_inches=0.1)
plt.show()

## ¿Quiénes aprobaron, desertaron y desaprobaron?



Se agrupan los usuarios ahora también por usuario, plataforma y actividad. Aquí se ve cuántas entregas hizo el usuario por tarea, y la calificación más alta en la misma.

In [ ]:
tareas_actividad_usuario = sub_file.groupby(['username', 'platform', 'actividad']).agg(
    submission_activity_user=('best', 'size'),
    max_grade=('grade', 'max')                  # valor mas alto de esas entregas
).reset_index()

In [ ]:
tareas_actividad_usuario.head(5)

Como la actividad de demo no tiene ningun peso en las calificaciones, se elimina.

In [ ]:
tareas_actividad_usuario['actividad'].value_counts()

In [ ]:
tareas_actividad_usuario = tareas_actividad_usuario[tareas_actividad_usuario['actividad'] != 'a0_demo']

In [ ]:
tareas_actividad_usuario['actividad'].value_counts()

- Algunas actividades eran muy complicadas, por lo que se re-plantearon y reemplazó por otros ejercicios, así que se dejaron opcionales. Las prácticas opcionales no tienen peso despues de creados los nuevos ejercicios
- Algunas actividades se cambiaron a cómo se presentaba el ejercicio por lo que se reemplazó completamente. Dejaron de tener peso despues de la llegada de los nuevos ejercicios

Esto ocurre con las tareas a1 a a5.
Hay usuarios que tienen ambas, esto se maneja de la siguiente manera:

- 'a1_variables' : se elimina cuando a1_saludar está presente.
- 'a2_cadenas_de_texto': se elimina cuando a2_gritar está presente.
- 'a3_operadores_numéricos':  se elimina cuando a3_dividir_en_grupos está presente.
- 'a4_ejemplo_1' : se elimina cuando a4_ejemplo_1_2 está presente.
- 'a5_tarea_1': se elimina cuando a5_tarea_1_2 está presente

In [ ]:

# {Actividad_Que_Gana: Actividad_Que_Se_Elimina}
reglas_eliminacion = {
    'a1_saludar': 'a1_variables',
    'a2_gritar': 'a2_cadenas_de_texto',
    'a3_dividir_en_grupos': 'a3_operadores_numericos',
    'a4_ejemplo_1_2': 'a4_ejemplo_1',
    'a5_tarea_1_2': 'a5_tarea_1'
}

tareas_actividad_usuario['prefijo'] = tareas_actividad_usuario['actividad'].str.extract(r'(a\d+)')

def limpiar_duplicados_especificos(grupo):
    if len(grupo) <= 1:
        return grupo

    actividades_presentes = set(grupo['actividad'].values)
    indices_a_quitar = []

    for ganadora, perdedora in reglas_eliminacion.items():
        if ganadora in actividades_presentes and perdedora in actividades_presentes:
            idx = grupo[grupo['actividad'] == perdedora].index
            indices_a_quitar.extend(idx.tolist())

    if indices_a_quitar:
        return grupo.drop(index=indices_a_quitar)

    return grupo

tareas_actividad_usuario_fixed = tareas_actividad_usuario.groupby(['username', 'prefijo'], group_keys=False).apply(limpiar_duplicados_especificos)
tareas_actividad_usuario_fixed = tareas_actividad_usuario_fixed.drop(columns=['prefijo'], errors='ignore')

In [ ]:
tareas_actividad_usuario_fixed[tareas_actividad_usuario_fixed['actividad'].str.contains('a4_ejemplo_1_2')]

In [ ]:
tareas_actividad_usuario_fixed[tareas_actividad_usuario_fixed['username'] == '']

In [ ]:
tareas_actividad_usuario_fixed['actividad'].value_counts()

Se comprueba que la limpieza de tareas repetidas haya quedado bien por usuario y plataforma.

In [ ]:
import re

def test_limpieza_bloques(df, numeros_bloques=[1, 2, 3, 4, 5]):
    print(f"{'BLOQUE':<10} | {'ESTADO':<15} | {'DETALLE'}")
    print("-" * 50)

    for n in numeros_bloques:
        patron = f'a{n}(?!\\d)'

        solo_bloque = df[df['actividad'].str.contains(patron, regex=True, na=False)]

        conteo_mismo_sitio = solo_bloque.groupby(['username', 'platform']).size()
        duplicados = conteo_mismo_sitio[conteo_mismo_sitio > 1]

        multi_plataforma = solo_bloque.groupby('username')['platform'].nunique()
        usuarios_multi = multi_plataforma[multi_plataforma > 1]

        bloque_label = f"a{n}"
        if duplicados.empty:
            estado = "LIMPIO"
            detalle = f"{len(usuarios_multi)} usuarios en multi-plataforma"
        else:
            estado = "ERROR"
            detalle = f"{len(duplicados)} casos de duplicados internos"

        print(f"{bloque_label:<10} | {estado:<15} | {detalle}")

test_limpieza_bloques(tareas_actividad_usuario_fixed)

In [ ]:
tareas_actividad_usuario_fixed.head(6)

UPDATE: 14  DE ABRIL

Para aprobar el curso se revisa que haya al menos un intento para la tarea a17  y tener un promedio, en el curso, superior al 0,6. ### Update: ya no se tiene en cuenta lo del la tarea a17 en el código de más adelante

Lo de la tarea a17 está relacionado a que el curso se va desbloqueando de forma incremental, por lo que quien tenga la tarea a17, es porque ha llegado al final.

De esta manera, se van a definir tres grupos de análisis:

- Aprobados : registraron tareas después de la a5 y su calificación igual o superior a 0,60

- Desertores : hicieron la primera unidad, es decir, de la tarea a1 a la a5 (cualquiera de las a5 que registre)

- Reprobados: registraron tareas después de la a5 pero su calificación no fue de más de 0,60



In [ ]:
tareas_por_usuario = tareas_actividad_usuario_fixed.groupby(['username', 'platform']).agg(
    actividades_hechas_por_usuario=('actividad', 'size'),
    nombres_actividades=('actividad', lambda x: list(x)) #
).reset_index()

In [ ]:
tareas_por_usuario.head(15)

In [ ]:
tareas_actividad_usuario_fixed.columns

In [ ]:
len(tareas_actividad_usuario_fixed)

Primero se crea el grupo de desertores: hicieron la primera unidad, es decir, de la tarea a1 a la a5 (cualquiera de las a5 que registre)

In [ ]:
def extraer_prefijo(actividad):
    match = re.search(r'a\d+', str(actividad))
    return match.group() if match else None

tareas_actividad_usuario_fixed['prefijo'] = tareas_actividad_usuario_fixed['actividad'].apply(extraer_prefijo)

In [ ]:
df_temp = tareas_actividad_usuario_fixed.copy()
df_temp['num_actividad'] = df_temp['actividad'].str.extract('(\d+)').astype(int)

#máxima actividad por usuario
progreso_por_dupla = df_temp.groupby(['username', 'platform'])['num_actividad'].max().reset_index()

desertores_unicos = progreso_por_dupla[progreso_por_dupla['num_actividad'] <= 5][['username', 'platform']]

# Se completa la informacion
full_info_desertores = tareas_actividad_usuario_fixed.merge(desertores_unicos, on=['username', 'platform'], how='inner')

Construyo un caso de prueba para revisar que esté tomando bien las tareas para los desertores.

In [ ]:
tareas_requeridas = {'a1', 'a2', 'a4'}
def solo_las_tres(grupo):
    return set(grupo['prefijo']) == tareas_requeridas

desertores_exactos = tareas_actividad_usuario_fixed.groupby(['username', 'platform']).filter(solo_las_tres)

In [ ]:
desertores_exactos

In [ ]:
tareas_actividad_usuario_fixed[tareas_actividad_usuario_fixed['username'] == ''] #modificado para proteccion de datos

In [ ]:
full_info_desertores[full_info_desertores['username'] == ''] #modificado para proteccion de datos

In [ ]:
len(full_info_desertores)

In [ ]:
len(desertores_unicos)

Contamos con:
- 2360 desertores únicos
- un total de 5548 submissions para ellos.

Quitamos estos desertores del df completo, ya que son un grupo aparte.

In [ ]:
df_aprobados_reprobados_full = tareas_actividad_usuario_fixed.merge(
    desertores_unicos,
    on=['username', 'platform'],
    how='left',
    indicator=True
)

df_aprobados_reprobados = df_aprobados_reprobados_full[df_aprobados_reprobados_full['_merge'] == 'left_only'].copy()

df_aprobados_reprobados.drop(columns=['_merge'], inplace=True)

Se realizan dos pruebas de código sobre el nuevo df df_aprobados_reprobados, para estar seguros de que allí no existen usuarios con menos de 5 tareas.

In [ ]:
total_original = len(tareas_actividad_usuario_fixed)
total_desertores = len(full_info_desertores)
total_activos = len(df_aprobados_reprobados)

print(f"Total original: {total_original}")
print(f"Suma de grupos: {total_desertores + total_activos}")

if total_original == (total_desertores + total_activos):
    print(" ¡Limpieza exitosa! Los grupos coinciden")

In [ ]:
df_temp_no = df_aprobados_reprobados.copy()
df_temp_no['num_actividad'] = df_temp_no['actividad'].str.extract('(\d+)').astype(int)

verificacion = df_temp_no.groupby(['username', 'platform'])['num_actividad'].max()

fallos = verificacion[verificacion <= 5]
print(f"Usuarios que no deberían estar aquí: {len(fallos)}")

In [ ]:
df_aprobados_reprobados.head(45)

In [ ]:
# ---- Abril 14: los aprobados y reprobados se calcularán sobre los que tienen de la tarea a5 en adelante, asi que este
# código queda deprecado
#usuarios_a17_keys = df_aprobados_reprobados[df_aprobados_reprobados['actividad'] == 'a17_tarea_3'][['username', 'platform']].drop_duplicates()
# df_con_a17 = df_aprobados_reprobados.merge(usuarios_a17_keys, on=['username', 'platform'], how='inner')
#df_temp = df_aprobados_reprobados.merge(usuarios_a17_keys, on=['username', 'platform'], how='left', indicator=True)
#print(f"Usuarios (username+platform) totales: {len(df_aprobados_reprobados[['username', 'platform']].drop_duplicates())}")
#print(f"Usuarios en grupo con a17: {len(usuarios_a17_keys)}")

#verificar_tareas_por_usuario = df_con_a17.groupby(['username', 'platform']).agg(
#    actividades_hechas_por_usuario=('actividad', 'size'),           # Esto reemplaza al .size()
#    nombres_actividades=('actividad', lambda x: list(x)) # Esto crea el array/lista
#).reset_index()

#def verificar_a17(lista_actividades):
#    prefijos_usuario = {re.search(r'a\d+', act).group() for act in lista_actividades if re.search(r'a\d+', act)}
#    return 'a17' in prefijos_usuario

#verificar_tareas_por_usuario['tiene_a17'] = verificar_tareas_por_usuario['nombres_actividades'].apply(verificar_a17)

# Usuarios que NO tienen la tarea a17
#faltantes_a17 = verificar_tareas_por_usuario[verificar_tareas_por_usuario['tiene_a17'] == False]
# print(f"Usuarios que no llegaron a la tarea a17: {len(faltantes_a17)}")

Los aprobados y reprobados se calcularán sobre los que tienen de la tarea a5 en adelante, calculando su nota final a partir de las actividades realizadas. Cada actividad tienen un peso, para sacar la nota final:


Existen dos tipos de actividades:
- Prácticas: Actividades cortas con un tema específico dentro de la unidad. Las practicas valen 30%
- Tarea: Actividad que aplica los principales temas vistos durante la unidad. Las tareas son el 60%

- La nota 10% restante corresponde a la autoevaluación. La autoevaluación si se completa se obtiene la puntuación completa.

Claridades:
- Las prácticas suman el 30%
- Las tareas suman el 60%. Siendo 3 tareas, cada una es el 20%, pero esto se divide:
Un ejemplo, en donde existe un video donde se muestra cómo resolver un ejercicio muy similar y simplificado que vale el 10%
- La actividad principal que vale 90%
- Hay un caso especial de un ejemplo en la versión 2 en donde por un error de la plataforma se tuvo que clonar el ejercicio

Así las actividades tienen los siguientes pesos:

Se hace un compendio final para tener los usuarios, sus fechas de actividad en días y el número de actividades entregadas

In [ ]:
pesos_actividades = {
    'a0_demo': 0,
    'a1_variables': 0.027,
    'a1_saludar': 0.027,
    'a2_cadenas_de_texto': 0.027,
    'a2_gritar': 0.027,
    'a3_operadores_numericos': 0.027,
    'a3_dividir_en_grupos': 0.027,
    'a4_ejemplo_1': 0.02,
    'a4_ejemplo_1_2': 0.02,
    'a5_tarea_1': 0.18,
    'a5_tarea_1_2': 0.18,
    'a6_cursores_y_areas': 0.027,
    'a7_bisisestos': 0.027,
    'a8_while': 0.027,
    'a9_for': 0.027,
    'a10_ejemplo_2': 0.02,
    'a11_tarea_2': 0.18,
    'a12_trozos_en_cadenas': 0.027,
    'a13_listas': 0.027,
    'a14_diccionarios': 0.027,
    'a15_conjuntos': 0.027,
    'a16_ejemplo_3': 0.02,
    'a17_tarea_3': 0.18
}

In [ ]:
df_aprobados_reprobados['peso'] = df_aprobados_reprobados['actividad'].map(pesos_actividades)

df_aprobados_reprobados['nota_ponderada'] = df_aprobados_reprobados['max_grade'] * df_aprobados_reprobados['peso']

In [ ]:
errores_mapeo = df_aprobados_reprobados[df_aprobados_reprobados['peso'].isna()]

if errores_mapeo.empty:
    print("Todas las actividades encontraron su peso.")
else:
    print("Hay actividades que no se mapearon correctamente:")
    print(errores_mapeo['actividad'].unique())

In [ ]:
df_aprobados_reprobados['nota_final'] = df_aprobados_reprobados.groupby(['username', 'platform'])['nota_ponderada'].transform('sum')


In [ ]:
df_aprobados_reprobados.columns

In [ ]:
full_info_aprobados = df_aprobados_reprobados[df_aprobados_reprobados['nota_final'] >= 60]
full_info_reprobados = df_aprobados_reprobados[df_aprobados_reprobados['nota_final'] < 60]

In [ ]:
aprobados_unicos = full_info_aprobados.drop_duplicates(subset=['username', 'platform'])
aprobados_unicos = aprobados_unicos[['username', 'platform','nota_final']]

In [ ]:
reprobados_unicos = full_info_reprobados.drop_duplicates(subset=['username', 'platform'])
reprobados_unicos = reprobados_unicos[['username', 'platform', 'nota_final']]

In [ ]:
len(aprobados_unicos)b #1549

In [ ]:
len(reprobados_unicos)

In [ ]:
len(desertores_unicos)

In [ ]:
len(aprobados_unicos)+len(reprobados_unicos)+len(desertores_unicos)

In [ ]:
5938-4848

## A partir de agrupar los usuarios y la plataforma, se ven que existen alumnos que han cursado más de una vez. Analizar los estudiantes que han cursado más de una vez el curso. Se pueden ver las fechas en las que empezaron y abandonaron el curso.

In [ ]:
submissions_user_counts = sub_file.groupby(['username', 'platform']).size().reset_index(name='count')

In [ ]:
usuarios_en_ambas = submissions_user_counts[submissions_user_counts.duplicated('username', keep=False)]

usuarios_en_ambas = usuarios_en_ambas.sort_values('username')

Hay 155 usuarios que están registrados dos veces

In [ ]:
len(usuarios_en_ambas)

In [ ]:
len(usuarios_en_ambas['username'].unique())

In [ ]:
usuarios_en_ambas_fechas = pd.merge(
        usuarios_en_ambas,
        sub_file,
        how='left',
        left_on=['username', 'platform'], right_on=['username', 'platform'], indicator=True)


In [ ]:
rango_actividad_reintentos = usuarios_en_ambas_fechas.groupby(['username', 'platform'])['solo_fecha'].agg(['min', 'max']).reset_index()


rango_actividad_reintentos.columns = ['username', 'platform', 'fecha_inicio', 'fecha_final']
rango_actividad_reintentos['fecha_inicio'] = pd.to_datetime(rango_actividad_reintentos['fecha_inicio'])
rango_actividad_reintentos['fecha_final'] = pd.to_datetime(rango_actividad_reintentos['fecha_final'])
rango_actividad_reintentos['dias_totales'] = (rango_actividad_reintentos['fecha_final'] - rango_actividad_reintentos['fecha_inicio']).dt.days

print(rango_actividad_reintentos)

In [ ]:
plt.figure(figsize=(12, 6))

sns.histplot(rango_actividad_reintentos['dias_totales'], bins=40, kde=True, color='darkslateblue', alpha=0.7)

plt.axvline(x=15, color='green', linestyle='--', label='Ritmo Constante (<15 días)')
plt.axvline(x=60, color='orange', linestyle='--', label='Reintentos Probables (>60 días)')

#plt.title('Distribución de Tiempo Total: Identificando Reintentos', fontsize=14)
plt.xlabel('Días Totales (Fecha Final - Fecha Inicio)', fontsize=12)
plt.ylabel('Cantidad de Estudiantes', fontsize=12)
nombre_archivo = os.path.join(imagenes_grupos_dir, "11_grafica_reintentos.pdf")
plt.savefig(nombre_archivo, format="pdf", bbox_inches="tight", pad_inches=0.1)
plt.legend()
plt.show()

Quiénes de los que cursaron dos veces, lograron aprobar el curso?

In [ ]:
reintentos_aprobados = pd.merge(
        rango_actividad_reintentos,
        aprobados_unicos,
        how='inner',
        left_on=['username', 'platform'], right_on=['username', 'platform'], indicator=True)


76 estudiantes volvieron y aprobaron el curso

In [ ]:
len(reintentos_aprobados)

In [ ]:
reintentos_aprobados

In [ ]:
df_duplicados = rango_actividad_reintentos.sort_values(['username', 'fecha_inicio'])

df_duplicados['n_intento'] = df_duplicados.groupby('username').cumcount() + 1

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 8))

for i, (user, grupo) in enumerate(df_duplicados.groupby('username').head(40).groupby('username')):
    for _, fila in grupo.iterrows():
        color = 'skyblue' if fila['n_intento'] == 1 else 'orange'
        plt.hlines(y=i, xmin=fila['fecha_inicio'], xmax=fila['fecha_final'],
                   colors=color, linewidth=6, label=f"Intento {fila['n_intento']}" if i == 0 else "")

##plt.title('Comparación de Periodos de Inscripción (Intento 1 vs 2)', fontsize=14)
plt.xlabel('Línea de Tiempo')
plt.ylabel('Estudiantes (Muestra)')
plt.legend(['Intento 1', 'Intento 2'], loc='upper right')
plt.grid(axis='x', linestyle='--', alpha=0.5)
nombre_archivo = os.path.join(imagenes_grupos_dir, "12_grafico_intento1_2.pdf")
plt.savefig(nombre_archivo, format="pdf", bbox_inches="tight", pad_inches=0.1)
plt.show()

## Análisis de submissions para los tres grupos (aprobados, reprobados, desertores)

In [ ]:
full_info_aprobados['grupo'] = 'aprobados'
full_info_reprobados['grupo'] = 'reprobados'
full_info_desertores['grupo'] = 'desertores'

In [ ]:
def extraer_prefijo(actividad):
    match = re.search(r'a\d+', str(actividad))
    return match.group() if match else None

full_info_aprobados['prefijo'] = full_info_aprobados['actividad'].apply(extraer_prefijo)
full_info_reprobados['prefijo'] = full_info_reprobados['actividad'].apply(extraer_prefijo)
full_info_desertores['prefijo'] = full_info_desertores['actividad'].apply(extraer_prefijo)

In [ ]:
to_analize_df = pd.concat([full_info_aprobados, full_info_reprobados, full_info_desertores])

In [ ]:
to_analize_df.columns

In [ ]:
colores_grupos = {
    'desertores': '#F1C40F',
    'aprobados': '#2ECC71',
    'reprobados': '#FF4B4B'
}

In [ ]:

orden_prefijos = sorted(to_analize_df['prefijo'].unique(),
                        key=lambda x: int(x.replace('a', '')))

plt.figure(figsize=(12, 6))
sns.countplot(x='prefijo', hue='grupo', data=to_analize_df, order=orden_prefijos, palette=colores_grupos)

#plt.title('Submissions por Tarea y Grupo')
plt.xticks(rotation=45)
plt.ylabel("Cantidad")
plt.xlabel("Tarea (alias)")


nombre_archivo = os.path.join(imagenes_grupos_dir, "1_graficos_submissions_tarea_grupo.pdf")
plt.savefig(nombre_archivo, format="pdf", bbox_inches="tight", pad_inches=0.1)
plt.show()

In [ ]:
orden_prefijos = sorted(to_analize_df['prefijo'].unique(),
                        key=lambda x: int(x.replace('a', '')))

df_counts = to_analize_df.groupby(['prefijo', 'grupo']).size().reset_index(name='cantidad_usuarios')

df_counts['prefijo'] = pd.Categorical(df_counts['prefijo'], categories=orden_prefijos, ordered=True)
df_counts = df_counts.sort_values('prefijo')

plt.figure(figsize=(12, 6))
sns.lineplot(data=df_counts, x='prefijo', y='cantidad_usuarios', hue='grupo', marker='o', linewidth=2,
             palette=colores_grupos)

#plt.title('Evolución por submissions y Grupo', fontsize=14)
plt.ylabel('Número de interacciones')
plt.xlabel('Tarea (alias)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.xticks(rotation=45)
nombre_archivo = os.path.join(imagenes_grupos_dir, "2_grafica_evolucion_submission_grupo.pdf")
plt.savefig(nombre_archivo, format="pdf", bbox_inches="tight", pad_inches=0.1)
plt.show()

In [ ]:
to_analize_df.columns

Primedio de veces que se entregó una tarea para cada uno de los grupos:

In [ ]:
sns.barplot(x='grupo', y='submission_activity_user', data=to_analize_df)
#plt.title('Esfuerzo Promedio (Intentos) por Grupo')
plt.ylabel("Número de intentos")
plt.xlabel("Grupo")
nombre_archivo = os.path.join(imagenes_grupos_dir, "3_grafica_esfuerzo_promedio.pdf")
plt.savefig(nombre_archivo, format="pdf", bbox_inches="tight", pad_inches=0.1)
plt.show()

Rango de días para los tres grupos:

In [ ]:
rango_actividad.columns

In [ ]:
aprobados_dias = pd.merge(
        rango_actividad,
        aprobados_unicos,
        how='inner',
        left_on=['username', 'platform'], right_on=['username', 'platform'], indicator=True)

reprobados_dias = pd.merge(
        rango_actividad,
        reprobados_unicos,
        how='inner',
        left_on=['username', 'platform'], right_on=['username', 'platform'], indicator=True)

desertores_dias = pd.merge(
        rango_actividad,
        desertores_unicos,
        how='inner',
        left_on=['username', 'platform'], right_on=['username', 'platform'], indicator=True)



In [ ]:
aprobados_dias['grupo'] = 'aprobados'
reprobados_dias['grupo'] = 'reprobados'
desertores_dias['grupo'] = 'desertores'

In [ ]:
len(desertores_dias)

In [ ]:
len(desertores_unicos)

In [ ]:
total_dias_df = pd.concat([aprobados_dias, reprobados_dias, desertores_dias])

In [ ]:
desertores_dias.columns

APROBADOS:
1549

REPROBADOS:
939

DESERTORES:
2360

In [ ]:
plt.figure(figsize=(8, 6))
sns.boxplot(
    data=total_dias_df,
    x='grupo',
    y='dias_totales',
    palette=colores_grupos
)
#plt.title('Días de Permanencia en el Curso por Grupo')
plt.ylabel('Días totales')
plt.xlabel("Grupos")
nombre_archivo = os.path.join(imagenes_grupos_dir, "5_grafica_dias_permanencia_curso.pdf")
plt.savefig(nombre_archivo, format="pdf", bbox_inches="tight", pad_inches=0.1)

plt.show()


In [ ]:
# medianas (Q2) y cuartiles por grupo
estadisticas = total_dias_df.groupby('grupo')['dias_totales'].agg(
    mediana='median',
    q1=lambda x: x.quantile(0.25),
    q3=lambda x: x.quantile(0.75),
    maximo='max'
)

print(estadisticas)

Promedios para los aprobados:

In [ ]:
media_dias = aprobados_dias['dias_totales'].mean()
print(f"Promedio de dias: {media_dias}")

min_dias = aprobados_dias['dias_totales'].min()
print(f"el que tardó menos dias: {min_dias}")

max_dias = aprobados_dias['dias_totales'].max()
print(f"más dias: {max_dias}")

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(aprobados_dias['dias_totales'], bins=15, kde=True, color='skyblue')

#plt.title('Distribución de Días para aprobar el curso', fontsize=15)
plt.xlabel('Días transcurridos (Desde 1ª hasta 17ª actividad)', fontsize=12)
plt.ylabel('Cantidad de estudiantes', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
nombre_archivo = os.path.join(imagenes_grupos_dir, "4_grafica_distribucion_dias.pdf")
plt.savefig(nombre_archivo, format="pdf", bbox_inches="tight", pad_inches=0.1)
plt.show()

Promedios para los reprobados:

In [ ]:
media_dias = reprobados_dias['dias_totales'].mean()
print(f"Promedio de dias: {media_dias}")

min_dias = reprobados_dias['dias_totales'].min()
print(f"el que tardó menos dias: {min_dias}")

max_dias = reprobados_dias['dias_totales'].max()
print(f"más dias: {max_dias}")

Promedios para los desertores

In [ ]:
media_dias = desertores_dias['dias_totales'].mean()
print(f"Promedio de dias: {media_dias}")

min_dias = desertores_dias['dias_totales'].min()
print(f"el que tardó menos dias: {min_dias}")

max_dias = desertores_dias['dias_totales'].max()
print(f"más dias: {max_dias}")

Hay personas que aprobaron pero entregaron todo un solo dia

In [ ]:
aprobados_dias[aprobados_dias['dias_totales'] == 0 ]

In [ ]:
plt.figure(figsize=(10, 6))
sns.kdeplot(
    data=total_dias_df,
    x='dias_totales',
    hue='grupo',
    palette=colores_grupos,
    fill=True,
    common_norm=False
)
#plt.title('Distribución de Permanencia (Densidad)')
plt.xlabel('Días desde el inicio')
plt.ylabel('Densidad')
nombre_archivo = os.path.join(imagenes_grupos_dir, "6_grafica_densidad_permanencia.pdf")
plt.savefig(nombre_archivo, format="pdf", bbox_inches="tight", pad_inches=0.1)
plt.show()

La gran mayoría de los desertores toman la decisión de irse casi de inmediato.

Las curvas verde (aprobados) y roja (reprobados) son casi idénticas en su forma y posición. El tiempo de permanencia puede no ser el factor diferenciador entre ganar o perder. Tanto los que aprueban como los que reprueban se quedan en el curso una cantidad de tiempo similar.

In [ ]:
df_zoom = total_dias_df[total_dias_df['dias_totales'] <= 60]

plt.figure(figsize=(12, 6))


sns.kdeplot(
    data=df_zoom,
    x='dias_totales',
    hue='grupo',
    palette=colores_grupos,
    fill=True,
    common_norm=False,
    bw_adjust=0.5
)

##plt.title('Zoom: Comportamiento en los Primeros 60 Días', fontsize=15)
plt.xlabel('Días desde el inicio')
plt.ylabel('Densidad de estudiantes')
plt.grid(True, linestyle=':', alpha=0.6)
plt.xlim(0, 60)
nombre_archivo = os.path.join(imagenes_grupos_dir, "7_grafica_zoom_permanencia.pdf")
plt.savefig(nombre_archivo, format="pdf", bbox_inches="tight", pad_inches=0.1)

plt.show()

In [ ]:
df_zoom.columns

In [ ]:
full_aprobados_dias = pd.merge(
        rango_actividad,
        full_info_aprobados,
        how='inner',
        left_on=['username', 'platform'], right_on=['username', 'platform'], indicator=True)

full_reprobados_dias = pd.merge(
        rango_actividad,
        full_info_reprobados,
        how='inner',
        left_on=['username', 'platform'], right_on=['username', 'platform'], indicator=True)

full_desertores_dias = pd.merge(
        rango_actividad,
        full_info_desertores,
        how='inner',
        left_on=['username', 'platform'], right_on=['username', 'platform'], indicator=True)



In [ ]:
len(full_aprobados_dias)

In [ ]:
len(full_info_aprobados)

In [ ]:
full_info_dias = pd.concat([full_desertores_dias, full_reprobados_dias, full_aprobados_dias])

In [ ]:
full_info_dias.columns

In [ ]:
df_solo_desertores = full_info_dias[full_info_dias['grupo'] == 'desertores'].copy()
df_solo_desertores['num_act'] = df_solo_desertores['prefijo'].str.extract('(\d+)').astype(int)

punto_abandono = df_solo_desertores.groupby(['username', 'platform'])['num_act'].max().reset_index()

conteo_abandono = punto_abandono['num_act'].value_counts().sort_index().reset_index()
conteo_abandono.columns = ['tarea', 'cantidad_usuarios']
conteo_abandono['tarea_nombre'] = conteo_abandono['tarea'].astype(str)


In [ ]:
nombres_tareas = conteo_abandono['tarea_nombre'].unique()
orden_logico = sorted(nombres_tareas, key=lambda x: int(x.replace('a', '')))

plt.figure(figsize=(12, 6))

ax = sns.barplot(
    data=conteo_abandono,
    x='tarea_nombre',
    y='cantidad_usuarios',
    order=orden_logico,
    palette="Reds_r"
)

#plt.title('¿En qué tarea se rinden los estudiantes?', fontsize=15)
plt.ylabel('Cantidad de desertores')
plt.xlabel('Última tarea realizada')
plt.grid(axis='y', linestyle='--', alpha=0.3)


for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(f'{int(height)}',
                    (p.get_x() + p.get_width() / 2., height),
                    ha = 'center', va = 'center',
                    xytext = (0, 9),
                    textcoords = 'offset points',
                    fontweight='bold')

nombre_archivo = os.path.join(imagenes_grupos_dir, "8_grafico_tarea_rinden.pdf")
plt.savefig(nombre_archivo, format="pdf", bbox_inches="tight", pad_inches=0.1)
plt.show()

In [ ]:
df_solo_desertores = full_info_dias[full_info_dias['grupo'] == 'reprobados'].copy()
df_solo_desertores['num_act'] = df_solo_desertores['prefijo'].str.extract('(\d+)').astype(int)

punto_abandono = df_solo_desertores.groupby(['username', 'platform'])['num_act'].max().reset_index()

conteo_abandono = punto_abandono['num_act'].value_counts().sort_index().reset_index()
conteo_abandono.columns = ['tarea', 'cantidad_usuarios']
conteo_abandono['tarea_nombre'] = conteo_abandono['tarea'].astype(str)


In [ ]:
nombres_tareas = conteo_abandono['tarea_nombre'].unique()
orden_logico = sorted(nombres_tareas, key=lambda x: int(x.replace('a', '')))

plt.figure(figsize=(12, 6))

ax = sns.barplot(
    data=conteo_abandono,
    x='tarea_nombre',
    y='cantidad_usuarios',
    order=orden_logico,
    palette="Reds_r"
)

#plt.title('¿En qué tarea se rinden los estudiantes (reprobados)?', fontsize=15)
plt.ylabel('Cantidad de reprobados')
plt.xlabel('Última tarea realizada')
plt.grid(axis='y', linestyle='--', alpha=0.3)


for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(f'{int(height)}',
                    (p.get_x() + p.get_width() / 2., height),
                    ha = 'center', va = 'center',
                    xytext = (0, 9),
                    textcoords = 'offset points',
                    fontweight='bold')

nombre_archivo = os.path.join(imagenes_grupos_dir, "9_grafica_tarea_rinden_reprobados.pdf")
plt.savefig(nombre_archivo, format="pdf", bbox_inches="tight", pad_inches=0.1)
plt.show()

La tarea 7

Velocidad: cuántas tareas completa un estudiante por cada día que permanece en el curso.

In [ ]:
progreso_conteo = full_info_dias.groupby(['username', 'platform'])['prefijo'].nunique().reset_index()
progreso_conteo.rename(columns={'prefijo': 'total_tareas'}, inplace=True)

info_usuarios = full_info_dias[['username', 'platform', 'grupo', 'dias_totales']].drop_duplicates()

df_velocidad = info_usuarios.merge(progreso_conteo, on=['username', 'platform'])

df_velocidad['velocidad'] = df_velocidad['total_tareas'] / (df_velocidad['dias_totales'] + 0.5)

In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    data=df_velocidad,
    x='grupo',
    y='velocidad',
    palette=colores_grupos,
    capsize=.1
)

#plt.title('Velocidad de Progreso Promedio (Tareas/Día)', fontsize=14)
plt.ylabel('Tareas completadas por día')
plt.xlabel('Grupos')
plt.grid(axis='y', linestyle='--', alpha=0.7)
nombre_archivo = os.path.join(imagenes_grupos_dir, "10_grafica_velocidad.pdf")
plt.savefig(nombre_archivo, format="pdf", bbox_inches="tight", pad_inches=0.1)
plt.show()

In [ ]:
flash_users = df_velocidad.sort_values(by='velocidad', ascending=False).head(5)
print("Usuarios con mayor velocidad de avance:")
print(flash_users[['username', 'total_tareas', 'dias_totales', 'velocidad']])

Se traerá la información completa de los submissions


In [ ]:
full_info_dias['submission_activity_user'].sum()

El número anterior es la cantidad de filas que debe tener el nuevo df que va a tener toda la informacion completa

In [ ]:
full_info_dias.drop(columns=['_merge'], inplace=True)

In [ ]:
full_grupos_x_submission = pd.merge(
        full_info_dias,
        sub_file,
        how='inner',
        left_on=['username', 'platform', 'actividad'], right_on=['username', 'platform', 'actividad'], indicator=True)



In [ ]:
len(full_grupos_x_submission)

In [ ]:
full_grupos_x_submission.columns

In [ ]:
full_grupos_x_submission[['username', 'platform',  '_id', 'archive',
       'best', 'custom[custom_additional_info]',
       'custom[custom_summary_result]', 'grade', 'input', 'is_late_submission',
       'outcome_consumer_key',
       'result', 'status']].head()

custom[custom_summary_result] para los tres grupos:

In [ ]:
full_grupos_x_submission['custom[custom_summary_result]'].value_counts()

In [ ]:
resumen_errores = full_grupos_x_submission.groupby(['grupo', 'custom[custom_summary_result]']).size().reset_index(name='conteo')

df_plot_errores = resumen_errores.pivot(index='grupo', columns='custom[custom_summary_result]', values='conteo').fillna(0)

cols = ['ACCEPTED', 'WRONG_ANSWER', 'RUNTIME_ERROR', 'TIME_LIMIT_EXCEEDED', 'PRESENTATION_ERROR']
df_plot_errores = df_plot_errores[[c for c in cols if c in df_plot_errores.columns]]

In [ ]:
df_perc = df_plot_errores.divide(df_plot_errores.sum(axis=1), axis=0) * 100

ax = df_perc.plot(kind='barh', stacked=True, figsize=(12, 7),
                  color=['#2ECC71', '#E74C3C', '#3498DB', '#9B59B6', '#F1C40F'])

plt.title('Composición de Resultados de Envío por Grupo (%)', fontsize=15)
plt.xlabel('Porcentaje del Total de Envíos')
plt.ylabel('Grupo')
plt.legend(title='Resultado', bbox_to_anchor=(1.05, 1), loc='upper left')

for p in ax.patches:
    width = p.get_width()
    if width > 5: # segmento grande
        ax.annotate(f'{width:.1f}%',
                    (p.get_x() + width/2, p.get_y() + p.get_height()/2),
                    ha='center', va='center', color='white', fontweight='bold')

plt.tight_layout()
plt.show()

Result para los tres grupos:

In [ ]:
full_grupos_x_submission['result'].value_counts()

In [ ]:
resumen_submissions = full_grupos_x_submission.groupby(['grupo', 'result']).size().reset_index(name='conteo')

df_plot = resumen_submissions.pivot(index='grupo', columns='result', values='conteo').fillna(0)

df_percent = df_plot.divide(df_plot.sum(axis=1), axis=0) * 100

cols_ordenadas = ['success', 'failed', 'crash', 'killed', 'overflow', 'timeout']
df_percent = df_percent[[c for c in cols_ordenadas if c in df_percent.columns]]

In [ ]:
plt.figure(figsize=(12, 7))
ax = df_percent.plot(kind='barh', stacked=True,
                      color=['#2ecc71', '#e74c3c', '#9b59b6', '#34495e', '#f1c40f', '#e67e22'],
                      figsize=(12, 7))

plt.title('Análisis de Submissions: Éxito vs. Fallo por Grupo', fontsize=15)
plt.xlabel('Porcentaje de Intentos (%)')
plt.ylabel('Grupo')
plt.legend(title='Resultado', bbox_to_anchor=(1.05, 1), loc='upper left')

for p in ax.patches:
    width = p.get_width()
    if width > 5: # segmento grande
        ax.annotate(f'{width:.1f}%',
                    (p.get_x() + width/2, p.get_y() + p.get_height()/2),
                    ha='center', va='center', color='white', fontweight='bold')

plt.tight_layout()
plt.show()

Como casi no se ven los grupos pequeños, hacemos un zoom

In [ ]:
errores_tecnicos = ['crash', 'killed', 'overflow', 'timeout']
df_solo_tech = full_grupos_x_submission[full_grupos_x_submission['result'].isin(errores_tecnicos)]

df_tech_counts = df_solo_tech.groupby(['grupo', 'result']).size().reset_index(name='conteo')

plt.figure(figsize=(12, 6))
ax = sns.barplot(
    data=df_tech_counts,
    x='result',
    y='conteo',
    hue='grupo',
    palette=colores_grupos
)

plt.yscale('log')

plt.title('Detalle de Errores Técnicos por Grupo (Escala Logarítmica)', fontsize=15)
plt.ylabel('Cantidad de Envíos (Escala Log)')
plt.xlabel('Tipo de Error Técnico')
plt.grid(True, axis='y', which="both", linestyle='--', alpha=0.3)

for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(f'{int(height)}',
                    (p.get_x() + p.get_width() / 2., height),
                    ha = 'center', va = 'bottom',
                    xytext = (0, 5), textcoords = 'offset points',
                    fontsize=9, fontweight='bold')

plt.show()

# Análisis de Caracterización - Submission

## Relacionar los que sí terminaron el curso con la caracterización

Aprovechando que anteriormente se tiene la lista de qué usuarios aprobaron el curso, veremos con de cuántos tenemos información de caracterización.


In [ ]:
len(aprobados_unicos)

In [ ]:
aprobados_unicos.head()

In [ ]:
total_curso_aprobado_char = pd.merge(
    aprobados_unicos,
    char_file,
    how='inner',
    left_on=['username', 'platform'],
    right_on=['username', 'platform'],
    indicator=False
)

print("match:", len(total_curso_aprobado_char))

In [ ]:
plt.figure(figsize=(10,6))
sns.countplot(data=total_curso_aprobado_char, y='ocupacion', order=char_file['ocupacion'].value_counts().index)

plt.title('Conteo de Ocupaciones para quienes aprobaron el curso')
plt.show()

In [ ]:
orden_deseado = ['1-2 horas', '3-4 horas', '5-6 horas', '7 horas o más']

plt.figure(figsize=(10,6))

sns.countplot(
    data=total_curso_aprobado_char,
    y='dedicacionsemanal',
    order=orden_deseado
)

plt.title('Dedicación semanal')
plt.ylabel(None)
plt.show()

In [ ]:
orden_deseado = ['basico', 'intermedio', 'avanzado', 'nulo']

plt.figure(figsize=(10,6))

sns.countplot(
    data=total_curso_aprobado_char,
    y='nivelprogramacion',
    order=orden_deseado
)

plt.title('Nivel de programación')
plt.ylabel(None)
plt.show()

In [ ]:
for col in ['dondeconocioelcurso', 'motivacion', 'ocupacion', 'lenguajes_programacion', 'nivelprogramacion', 'dedicacionsemanal']: print(f"=== {col} ===\n{total_curso_aprobado_char[col].apply(lambda x: tuple(x) if isinstance(x, list) else x).value_counts()}\n")

In [ ]:
stopwords_es_manual = [
     'ninguno', 'ninguna', 'aplica', 'solo', 'programación', 'nunca', 'programado',
    'primera', 'vez', 'algun', 'lenguaje', 'usado', 'es_spam_ruido', 'declaro_sin_experiencia',
    'campo_corregido', 'ninguno', 'ninguna', 'aplica', 'curso',
    'programacion', 'colombia', 'programación',
    'computadores', 'campo_corregido', 'declaro_sin_experiencia', 'ninguno', 'ninguna', 'general', 'declaro_sin_experiencia',
     'quiero', 'aprender', 'programacion', 'programación', 'deseo',
    'tener', 'poder', 'programar', 'mundo',
    'realizar', 'interesa', 'mas', 'nacional', 'colombia'
]

columnas_texto = ['razon_realizar_curso', 'otra_motivacion', 'otra_ocupacion', 'otro_lenguaje', 'otro_medio']
resultados_bigramas = {}

for col in columnas_texto:
    total_curso_aprobado_char[col] = total_curso_aprobado_char[col].astype(str).fillna('')

    print(f"=== Procesando bigramas para: {col} ===")
    try:
        resultados_bigramas[col] = main_get_bigrams(col, total_curso_aprobado_char, stopwords_es_manual)
    except Exception as e:
        print(f"Error al procesar la columna {col}: {e}")

In [ ]:
total_curso_aprobado_char.columns

## Relacionar los que desertaron

In [ ]:
len(desertores_unicos)

In [ ]:
total_curso_desertores_char = pd.merge(
    desertores_unicos,
    char_file,
    how='inner',
    left_on=['username', 'platform'],
    right_on=['username', 'platform'],
    indicator=False
)

print("match:", len(total_curso_desertores_char))

In [ ]:
for col in ['dondeconocioelcurso', 'motivacion', 'ocupacion', 'lenguajes_programacion', 'nivelprogramacion', 'dedicacionsemanal']: print(f"=== {col} ===\n{total_curso_desertores_char[col].apply(lambda x: tuple(x) if isinstance(x, list) else x).value_counts()}\n")

In [ ]:
stopwords_es_manual = [
     'ninguno', 'ninguna', 'aplica', 'solo', 'programación', 'nunca', 'programado',
    'primera', 'vez', 'algun', 'lenguaje', 'usado', 'es_spam_ruido', 'declaro_sin_experiencia',
    'campo_corregido', 'ninguno', 'ninguna', 'aplica', 'curso',
    'programacion', 'colombia', 'programación',
    'computadores', 'campo_corregido', 'declaro_sin_experiencia', 'ninguno', 'ninguna', 'general', 'declaro_sin_experiencia',
     'quiero', 'aprender', 'programacion', 'programación', 'deseo',
    'tener', 'poder', 'programar', 'mundo',
    'realizar', 'interesa', 'mas', 'nacional', 'colombia'
]

columnas_texto = ['razon_realizar_curso', 'otra_motivacion', 'otra_ocupacion', 'otro_lenguaje', 'otro_medio']
resultados_bigramas = {}

for col in columnas_texto:
    total_curso_desertores_char[col] = total_curso_desertores_char[col].astype(str).fillna('')

    print(f"=== Procesando bigramas para: {col} ===")
    try:
        resultados_bigramas[col] = main_get_bigrams(col, total_curso_desertores_char, stopwords_es_manual)
    except Exception as e:
        print(f"Error al procesar la columna {col}: {e}")

In [ ]:
plt.figure(figsize=(10,6))
sns.countplot(data=total_curso_desertores_char, y='ocupacion', order=char_file['ocupacion'].value_counts().index)

plt.title('Conteo de Ocupaciones para quienes desertaron el curso')
plt.show()

In [ ]:
orden_deseado = ['1-2 horas', '3-4 horas', '5-6 horas', '7 horas o más']

plt.figure(figsize=(10,6))

sns.countplot(
    data=total_curso_desertores_char,
    y='dedicacionsemanal',
    order=orden_deseado
)

plt.title('Dedicación semanal de los desertores')
plt.ylabel(None)
plt.show()

In [ ]:
orden_deseado = ['basico', 'intermedio', 'avanzado', 'nulo']

plt.figure(figsize=(10,6))

sns.countplot(
    data=total_curso_desertores_char,
    y='nivelprogramacion',
    order=orden_deseado
)

plt.title('Nivel de programación de los desertores')
plt.ylabel(None)
plt.show()

## Relacionar los que reprobaron

In [ ]:
total_curso_reprobado_char = pd.merge(
    reprobados_unicos,
    char_file,
    how='inner',
    left_on=['username', 'platform'],
    right_on=['username', 'platform'],
    indicator=False
)

print("match:", len(total_curso_reprobado_char))

In [ ]:
for col in ['dondeconocioelcurso', 'motivacion', 'ocupacion', 'lenguajes_programacion', 'nivelprogramacion', 'dedicacionsemanal']: print(f"=== {col} ===\n{total_curso_reprobado_char[col].apply(lambda x: tuple(x) if isinstance(x, list) else x).value_counts()}\n")

In [ ]:
stopwords_es_manual = [
     'ninguno', 'ninguna', 'aplica', 'solo', 'programación', 'nunca', 'programado',
    'primera', 'vez', 'algun', 'lenguaje', 'usado', 'es_spam_ruido', 'declaro_sin_experiencia',
    'campo_corregido', 'ninguno', 'ninguna', 'aplica', 'curso',
    'programacion', 'colombia', 'programación',
    'computadores', 'campo_corregido', 'declaro_sin_experiencia', 'ninguno', 'ninguna', 'general', 'declaro_sin_experiencia',
     'quiero', 'aprender', 'programacion', 'programación', 'deseo',
    'tener', 'poder', 'programar', 'mundo',
    'realizar', 'interesa', 'mas', 'nacional', 'colombia'
]

columnas_texto = ['razon_realizar_curso', 'otra_motivacion', 'otra_ocupacion', 'otro_lenguaje', 'otro_medio']
resultados_bigramas = {}

for col in columnas_texto:
    total_curso_reprobado_char[col] = total_curso_reprobado_char[col].astype(str).fillna('')

    print(f"=== Procesando bigramas para: {col} ===")
    try:
        resultados_bigramas[col] = main_get_bigrams(col, total_curso_reprobado_char, stopwords_es_manual)
    except Exception as e:
        print(f"Error al procesar la columna {col}: {e}")

In [ ]:
plt.figure(figsize=(10,6))
sns.countplot(data=total_curso_reprobado_char, y='ocupacion', order=char_file['ocupacion'].value_counts().index)

plt.title('Conteo de Ocupaciones para quienes no aprobaron el curso')
plt.show()

In [ ]:
orden_deseado = ['1-2 horas', '3-4 horas', '5-6 horas', '7 horas o más']

plt.figure(figsize=(10,6))

sns.countplot(
    data=total_curso_reprobado_char,
    y='dedicacionsemanal',
    order=orden_deseado
)

plt.title('Dedicación semanal de los que reprobaron el curso')
plt.ylabel(None)
plt.show()

In [ ]:
orden_deseado = ['basico', 'intermedio', 'avanzado', 'nulo']

plt.figure(figsize=(10,6))

sns.countplot(
    data=total_curso_reprobado_char,
    y='nivelprogramacion',
    order=orden_deseado
)

plt.title('Nivel de programación de los que reprobaron')
plt.ylabel(None)
plt.show()

## Caracterización y usuarios con un solo intento

In [ ]:
submissions_user_counts = sub_file.groupby(['username', 'platform']).size().reset_index(name='count')

In [ ]:
unique_once = submissions_user_counts[submissions_user_counts['count'] == 1]
len(unique_once)

In [ ]:
one_attemp_char = pd.merge(
    unique_once,
    char_file,
    how='inner',
    left_on=['username', 'platform'],
    right_on=['username', 'platform'],
    indicator=False
)

print("match:", len(one_attemp_char))

In [ ]:
for col in ['dondeconocioelcurso', 'motivacion', 'ocupacion', 'lenguajes_programacion', 'nivelprogramacion', 'dedicacionsemanal']: print(f"=== {col} ===\n{one_attemp_char[col].apply(lambda x: tuple(x) if isinstance(x, list) else x).value_counts()}\n")

Solo hay 123 alumnos que enviaron un solo attemp de los que podemos saber algo por su caracterización

In [ ]:
one_attemp_char.columns

In [ ]:
one_attemp_char['nivelprogramacion'].value_counts()

In [ ]:
one_attemp_char['motivacion'].value_counts()

In [ ]:
one_attemp_char['lenguajes_programacion'].value_counts()

In [ ]:
one_attemp_char['dedicacionsemanal'].value_counts()

In [ ]:
one_attemp_char['razon_realizar_curso']

# Análisis de submissions y autoevaluación

## Relacionar autoevaluación con quienes aprobaron el curso

In [ ]:
len(aprobados_unicos)

Tenemos 250 personas que aprobaron el curso y contamos con información de su autoevaluación

Update abril 14: ahora son 556

In [ ]:
total_curso_aprobado_eval = pd.merge(
    aprobados_unicos,
    eval_file,
    how='inner',
    left_on=['username', 'platform'],
    right_on=['username', 'platform'],
    indicator=False
)

print("match:", len(total_curso_aprobado_eval))

In [ ]:
fig, ax = survey_latex(get_results_types(total_curso_aprobado_eval, questions_dict_type_1, orden_deseado_type_1), orden_deseado_type_1)
fig.show()

In [ ]:
fig, ax = survey_latex_long_text(
    get_results_types(total_curso_aprobado_eval, questions_dict_type_2, orden_deseado_type_2),
    orden_deseado_type_2
)

fig.show()

Tenemos 9 personas que reprobaron el curso y contamos con información de su autoevaluación

Update abril 14: 33 personas

In [ ]:
total_curso_reprobado_eval = pd.merge(
    reprobados_unicos,
    eval_file,
    how='inner',
    left_on=['username', 'platform'],
    right_on=['username', 'platform'],
    indicator=False
)

print("match:", len(total_curso_reprobado_eval))

In [ ]:
fig, ax = survey_latex(get_results_types(total_curso_reprobado_eval, questions_dict_type_1, orden_deseado_type_1), orden_deseado_type_1)
fig.show()


In [ ]:
fig, ax = survey_latex_long_text(
    get_results_types(total_curso_reprobado_eval, questions_dict_type_2, orden_deseado_type_2),
    orden_deseado_type_2
)

fig.show()

Hay 12 personas desertoras con autoevaluación (considero que son personas que cursaron más de una vez el curso y esa autoevaluacion es de ese caso)

In [ ]:
total_curso_desertor_eval = pd.merge(
    desertores_unicos,
    eval_file,
    how='inner',
    left_on=['username', 'platform'],
    right_on=['username', 'platform'],
    indicator=False
)

print("match:", len(total_curso_desertor_eval))

In [ ]:
test = pd.merge(
    desertores_unicos,
    total_curso_aprobado_eval,
    how='inner',
    left_on=['username'],
    right_on=['username'],
    indicator=False
)


print("match:", len(test))

### APROBADOS:

In [ ]:
stopwords_es_manual = [
    'mayor', 'fortaleza', 'cada', 'tema', 'gran', 'ayuda',
    'proceso', 'aprendizaje', 'bien', 'curso', 'forma',
    'modulo', 'manera'
]
get_top_15("pregunta15", total_curso_aprobado_eval, stopwords_es_manual)

In [ ]:
stopwords_es_manual = [
    'mayor', 'fortaleza', 'cada', 'tema', 'gran', 'ayuda',
    'proceso', 'aprendizaje', 'bien', 'curso', 'forma',
    'modulo', 'manera'
]
get_top_15("pregunta16", total_curso_aprobado_eval, stopwords_es_manual)

In [ ]:
stopwords_es_manual = [
    'mayor', 'fortaleza', 'cada', 'tema', 'gran', 'ayuda',
    'proceso', 'aprendizaje', 'bien', 'curso', 'forma',
    'modulo', 'manera'
]
main_get_bigrams("pregunta15", total_curso_aprobado_eval, stopwords_es_manual)

In [ ]:
stopwords_es_manual = [
    'mayor', 'fortaleza', 'cada', 'tema', 'gran', 'ayuda',
    'proceso', 'aprendizaje', 'bien', 'curso', 'forma',
    'modulo', 'manera'
]
main_get_bigrams("pregunta16", total_curso_aprobado_eval, stopwords_es_manual)

In [ ]:
stopwords_es_manual = [
    'mayor', 'fortaleza', 'cada', 'tema', 'gran', 'ayuda',
    'proceso', 'aprendizaje', 'bien', 'curso', 'forma',
    'modulo', 'manera'
]
main_get_topics("pregunta15", total_curso_aprobado_eval, stopwords_es_manual)

In [ ]:
stopwords_es_manual = [
    'mayor', 'fortaleza', 'cada', 'tema', 'gran', 'ayuda',
    'proceso', 'aprendizaje', 'bien', 'curso', 'forma',
    'modulo', 'manera'
]
main_get_topics("pregunta16", total_curso_aprobado_eval, stopwords_es_manual)

In [ ]:
py_sentimento_raw_results = total_curso_aprobado_eval['pregunta15'].apply(
    lambda x: sentiment_analyzer.predict(str(x))
).tolist()

### REPROBADOS:

In [ ]:
stopwords_es_manual = [
    'mayor', 'fortaleza', 'cada', 'tema', 'gran', 'ayuda',
    'proceso', 'aprendizaje', 'bien', 'curso', 'forma',
    'modulo', 'manera'
]
get_top_15("pregunta15", total_curso_reprobado_eval, stopwords_es_manual)

In [ ]:
stopwords_es_manual = [
    'mayor', 'fortaleza', 'cada', 'tema', 'gran', 'ayuda',
    'proceso', 'aprendizaje', 'bien', 'curso', 'forma',
    'modulo', 'manera'
]
get_top_15("pregunta16", total_curso_reprobado_eval, stopwords_es_manual)

In [ ]:
stopwords_es_manual = [
    'mayor', 'fortaleza', 'cada', 'tema', 'gran', 'ayuda',
    'proceso', 'aprendizaje', 'bien', 'curso', 'forma',
    'modulo', 'manera'
]
main_get_bigrams("pregunta15", total_curso_reprobado_eval, stopwords_es_manual)

In [ ]:
stopwords_es_manual = [
    'mayor', 'fortaleza', 'cada', 'tema', 'gran', 'ayuda',
    'proceso', 'aprendizaje', 'bien', 'curso', 'forma',
    'modulo', 'manera'
]
main_get_bigrams("pregunta16", total_curso_reprobado_eval, stopwords_es_manual)

In [ ]:
stopwords_es_manual = [
    'mayor', 'fortaleza', 'cada', 'tema', 'gran', 'ayuda',
    'proceso', 'aprendizaje', 'bien', 'curso', 'forma',
    'modulo', 'manera'
]
main_get_topics("pregunta15", total_curso_reprobado_eval, stopwords_es_manual)

In [ ]:
stopwords_es_manual = [
    'mayor', 'fortaleza', 'cada', 'tema', 'gran', 'ayuda',
    'proceso', 'aprendizaje', 'bien', 'curso', 'forma',
    'modulo', 'manera'
]
main_get_topics("pregunta16", total_curso_reprobado_eval, stopwords_es_manual)

In [ ]:
data_list = []

for item in py_sentimento_raw_results:
    fila = {
        'sentimiento_final': item.output,
        **item.probas
    }
    data_list.append(fila)

df_resultados_aprobados_pregunta15_py = pd.DataFrame(data_list)

In [ ]:
print(df_resultados_aprobados_pregunta15_py['sentimiento_final'].value_counts())

# Análisis de Caracterización - Submission y Autoevaluacion

## Qué particularidades tienen los estudiantes que aprobaron, abandonaron y reprobaron el curso

Contamos con 167 alumnos que aprobaron el curso y que se les tiene toda su traza completa!

Update abril 14: son 305

In [ ]:
total_curso_aprobado_char = pd.merge(
    aprobados_unicos,
    char_file,
    how='inner',
    left_on=['username', 'platform'],
    right_on=['username', 'platform'],
    indicator=False
)

print("match:", len(total_curso_aprobado_char))

In [ ]:
total_curso_aprobado_char_eval = pd.merge(
    total_curso_aprobado_char,
    eval_file,
    how='inner',
    left_on=['username', 'platform'],
    right_on=['username', 'platform'],
    indicator=False
)

print("match:", len(total_curso_aprobado_char_eval))

In [ ]:
total_curso_aprobado_char_eval.columns

In [ ]:
total_curso_desaprobado_char = pd.merge(
    reprobados_unicos,
    char_file,
    how='inner',
    left_on=['username', 'platform'],
    right_on=['username', 'platform'],
    indicator=False
)

print("match:", len(total_curso_desaprobado_char))

In [ ]:
total_curso_desaprobado_char_eval = pd.merge(
    total_curso_desaprobado_char,
    eval_file,
    how='inner',
    left_on=['username', 'platform'],
    right_on=['username', 'platform'],
    indicator=False
)

print("match:", len(total_curso_desaprobado_char_eval))

In [ ]:
total_curso_desertor_char = pd.merge(
    desertores_unicos,
    char_file,
    how='inner',
    left_on=['username', 'platform'],
    right_on=['username', 'platform'],
    indicator=False
)

print("match:", len(total_curso_desertor_char))

In [ ]:
total_curso_desertor_char_eval = pd.merge(
    total_curso_desertor_char,
    eval_file,
    how='inner',
    left_on=['username', 'platform'],
    right_on=['username', 'platform'],
    indicator=False
)

print("match:", len(total_curso_desertor_char_eval))

In [ ]:
total_curso_desertor_char_eval

In [ ]:
total_curso_desertor_char_eval.columns

Se tienen 167 alumnos aprobados con toda su traza completa, 15 que desertaron y 8 que reprobaron


Update:
305 aprobados
25 que desertaron
8 reprobados

UPDATE 8 DE AGOSTO: Se juntarán los desertores con los reprobados para solo tener dos grupos de análisis

Análisis de preguntas de la autoevaluación con quienes aprobaron el curso

In [ ]:
total_curso_aprobado_char_eval['estado'] = 'Aprobado'
total_curso_desertor_char_eval['estado'] = 'Reprobado'  
total_curso_desaprobado_char_eval['estado'] = 'Reprobado'

df_reprobados_totales_char_eval = pd.concat(
    [total_curso_desertor_char_eval, total_curso_desaprobado_char_eval],
    ignore_index=True
)

df_total = pd.concat(
    [total_curso_aprobado_char_eval, df_reprobados_totales_char_eval],
    ignore_index=True
)

mapeo_likert = {
    '1-totalmente en desacuerdo': 1,
    '2-en desacuerdo': 2,
    '3-algo en desacuerdo': 3,
    '4-algo de acuerdo': 4,
    '5-de acuerdo': 5,
    '6-totalmente de acuerdo': 6,
    '0-no aplica (no conozco esta funcionalidad)': np.nan
}

questions_dict_type_1 = {
    "pregunta1" : "Mejora en programacion",
    "pregunta2" : "Notebooks de Python",
    "pregunta3" : "Pruebas de código",
    "pregunta4" : "Retroalimentación",
    "pregunta11" : "Uso de pistas",
    "pregunta12" : "Utilidad uncode",
    "pregunta13" : "Videos",
    "pregunta14" : "Visualización de código",
    "pregunta18" : "Foro",
}

def procesar_encuesta(df_grupo):
    df_res = df_grupo[list(questions_dict_type_1.keys())].copy()
    for col in df_res.columns:
        df_res[col] = df_res[col].astype(str).str.strip().map(mapeo_likert)
    return df_res.rename(columns=questions_dict_type_1)

df_aprob_num = procesar_encuesta(total_curso_aprobado_char_eval)
df_reprob_num = procesar_encuesta(df_reprobados_totales_char_eval)  # desertores + reprobados

In [ ]:
media_aprob = df_aprob_num.mean()
media_reprob = df_reprob_num.mean()

tabla_percepcion = pd.DataFrame({
    'Satisfacción Aprobados': media_aprob,
    'Satisfacción Reprobados': media_reprob
})

tabla_percepcion['Brecha'] = tabla_percepcion['Satisfacción Aprobados'] - tabla_percepcion['Satisfacción Reprobados']

print("COMPARATIVA DE PERCEPCIÓN POR GRUPO (Escala 1-6)")
display(tabla_percepcion.sort_values(by='Brecha', ascending=False).round(2))

Prueba estadística para hallar diferencia significativa

In [ ]:
import pandas as pd
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

resultados = []

for col in df_aprob_num.columns:
    aprobados_resp = df_aprob_num[col].dropna()
    reprobados_resp = df_reprob_num[col].dropna()

    # Prueba U de Mann-Whitney
    stat, p_val = mannwhitneyu(aprobados_resp, reprobados_resp, alternative='two-sided')

    brecha = aprobados_resp.mean() - reprobados_resp.mean()

    resultados.append({
        'Pregunta': col,
        'Media Aprobados': aprobados_resp.mean(),
        'Media Reprobados': reprobados_resp.mean(),
        'Brecha': brecha,
        'Estadístico U': stat,
        'p-valor sin corregir': p_val
    })

df_stats = pd.DataFrame(resultados)

# corrección por comparaciones múltiples (Benjamini-Hochberg)
_, p_adjusted, _, _ = multipletests(df_stats['p-valor sin corregir'], method='fdr_bh')
df_stats['p-valor ajustado (FDR)'] = p_adjusted

df_stats['Significativo (α=0.05)'] = df_stats['p-valor ajustado (FDR)'] < 0.05

df_stats = df_stats.sort_values(by='p-valor ajustado (FDR)').round(4)

print("RESULTADOS DE LA PRUEBA U DE MANN-WHITNEY")
display(df_stats)

Escala Likert 2

In [ ]:
total_curso_aprobado_char_eval['estado'] = 'Aprobado'
total_curso_desertor_char_eval['estado'] = 'Reprobado'
total_curso_desaprobado_char_eval['estado'] = 'Reprobado'

df_reprobados_totales_char_eval = pd.concat(
    [total_curso_desertor_char_eval, total_curso_desaprobado_char_eval],
    ignore_index=True
)

df_total = pd.concat(
    [total_curso_aprobado_char_eval, df_reprobados_totales_char_eval],
    ignore_index=True
)

mapeo_likert = {
    '1-muy bajo': 1,
    '2-bajo': 2,
    '3-medio': 3,
    '4-alto': 4,
    '5-muy alto': 5
}

questions_dict_type_2 = {
    "pregunta5" : "Abstraer un problema en forma de algoritmo e implementarlo en un programa con el lenguaje de programación Python.",
    "pregunta6" : "Conocer y aplicar los principios de manejo de variables, tipos de datos y operaciones entre valores numéricos y de texto en Python.",
    "pregunta7" : "Diseñar e implementar algoritmos con decisiones basadas en la evaluación de expresiones booleanas.",
    "pregunta8" : "Diseñar e implementar algoritmos con la aplicación de ciclos iterativos.",
    "pregunta9" : "Distinguir y elegir la colección apropiada para la resolución de un problema con una estructura de datos compleja.",
    "pregunta10" : "Usar colecciones en el diseño e implementación de algoritmos para la solución de problemas.",
    "pregunta17" : "Declarar y manipular listas, tuplas, matrices, diccionarios y conjuntos con el lenguaje de programación Python.",
}

# Función de procesamiento
def procesar_encuesta(df_grupo):
    df_res = df_grupo[list(questions_dict_type_2.keys())].copy()
    for col in df_res.columns:
        df_res[col] = df_res[col].astype(str).str.strip().map(mapeo_likert)
    return df_res.rename(columns=questions_dict_type_2)

df_aprob_num = procesar_encuesta(total_curso_aprobado_char_eval)
df_reprob_num = procesar_encuesta(df_reprobados_totales_char_eval)


In [ ]:

media_aprob = df_aprob_num.mean()
media_reprob = df_reprob_num.mean()

tabla_percepcion_type_2 = pd.DataFrame({
    'Satisfacción Aprobados': media_aprob,
    'Satisfacción Reprobados': media_reprob
})

tabla_percepcion_type_2['Brecha'] = (
    tabla_percepcion_type_2['Satisfacción Aprobados'] - tabla_percepcion_type_2['Satisfacción Reprobados']
)

print("COMPARATIVA DE PERCEPCIÓN POR GRUPO (Escala 1-5)")
display(tabla_percepcion_type_2.sort_values(by='Brecha', ascending=False).round(2))

Prueba estadística para hallar diferencias significativas

In [ ]:
import pandas as pd
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

resultados = []

for col in df_aprob_num.columns:
    aprobados_resp = df_aprob_num[col].dropna()
    reprobados_resp = df_reprob_num[col].dropna()

    stat, p_val = mannwhitneyu(aprobados_resp, reprobados_resp, alternative='two-sided')

    brecha = aprobados_resp.mean() - reprobados_resp.mean()

    resultados.append({
        'Pregunta': col,
        'Media Aprobados': aprobados_resp.mean(),
        'Media Reprobados': reprobados_resp.mean(),
        'Brecha': brecha,
        'Estadístico U': stat,
        'p-valor sin corregir': p_val
    })

df_stats = pd.DataFrame(resultados)

_, p_adjusted, _, _ = multipletests(df_stats['p-valor sin corregir'], method='fdr_bh')
df_stats['p-valor ajustado (FDR)'] = p_adjusted

df_stats['Significativo (α=0.05)'] = df_stats['p-valor ajustado (FDR)'] < 0.05

df_stats = df_stats.sort_values(by='p-valor ajustado (FDR)').round(4)

print("RESULTADOS DE LA PRUEBA U DE MANN-WHITNEY")
display(df_stats)

Análisis del nivel de programación de quienes aprobaron vs los que no aprobaron

In [ ]:
total_curso_aprobado_char_eval['estado'] = 'Aprobado'
total_curso_desertor_char_eval['estado'] = 'Desertor'
total_curso_desaprobado_char_eval['estado'] = 'Reprobado'
df_total = pd.concat([total_curso_aprobado_char_eval, total_curso_desertor_char_eval, total_curso_desaprobado_char_eval])

orden_nivel = df_total['nivelprogramacion'].unique()


In [ ]:
orden_nivel

In [ ]:
plt.figure(figsize=(12, 6))

ax = sns.countplot(data=df_total,
                   x='nivelprogramacion',
                   hue='estado',
                   palette={
                       'Aprobado': '#2ecc71',   # Verde
                       'Reprobado': '#e74c3c',  # Rojo
                       'Desertor': '#f39c12'    # Naranja/Amarillo
                   },
                   order=orden_nivel)

plt.xticks(ticks=range(len(orden_nivel)), labels=['Básico', 'Nulo', 'Intermedio', 'Avanzado'])

plt.ylabel('Número de estudiantes')
plt.xlabel('Nivel de programación inicial')
plt.legend(title='Estado Final')
plt.grid(axis='y', alpha=0.3)

nombre_archivo = os.path.join(imagenes_grupos_dir, "13_grafico_nivel_grupos.pdf")
plt.savefig(nombre_archivo, format="pdf", bbox_inches="tight", pad_inches=0.1)

plt.show()

In [ ]:
plt.figure(figsize=(12, 8))

paleta_estados = {
    'Aprobado': '#2ecc71',
    'Reprobado': '#e74c3c',
    'Desertor': '#f39c12'
}

mapeo_ocupacion = {
    'estudiante': 'Estudiante',
    'empleado': 'Empleado',
    'independiente': 'Independiente',
    'otro': 'Otro',
    'tareas_del_hogar': 'Tareas del hogar',
    'pensionado': 'Pensionado',
    'desempleado': 'Desempleado'
}

df_total['ocupacion_fmt'] = df_total['ocupacion'].map(lambda x: mapeo_ocupacion.get(x, str(x).capitalize()))

orden_ocupacion = df_total['ocupacion_fmt'].value_counts().index

sns.countplot(data=df_total,
              y='ocupacion_fmt',
              hue='estado',
              palette=paleta_estados,
              order=orden_ocupacion)

plt.xlabel('Número de estudiantes')
plt.ylabel('Ocupación')
plt.legend(title='Estado Final', loc='lower right')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()

nombre_archivo = os.path.join(imagenes_grupos_dir, "14_grafico_impacto_ocupacion.pdf")
plt.savefig(nombre_archivo, format="pdf", bbox_inches="tight", pad_inches=0.1)

plt.show()

## Comparar la dedicación semanal indicada en la caracterización con la indicada en la autoevaluación (para todos)

In [ ]:
full_file_path = '/content/drive/MyDrive/estudio/UNAL/new_project/data/generated/standardize/standardized-total-char-submissions-eval-file.csv'


In [ ]:
full_file = pd.read_csv(full_file_path)


In [ ]:
len(full_file)

In [ ]:
full_user_counts = full_file.groupby(['username', 'platform']).size().reset_index(name='count')

In [ ]:
len(full_file)

In [ ]:
full_file.columns

In [ ]:
full_file[['username', 'platform', 'dedicacionsemanal', 'pregunta19']]

In [ ]:
full_file['pregunta19'].unique()

El siguiente análisis se realiza sobre todos los estudiantes que tienen su traza completa, sin importar si aprobaron o no el curso.

A continuación de ve una matriz de confusión que permite ver cómo cambió la deducación semanal (pre y post)

In [ ]:
df_to_analyze = full_file[['username', 'platform', 'dedicacionsemanal', 'pregunta19']]
df_to_analyze = df_to_analyze.groupby(['username', 'platform', 'dedicacionsemanal', 'pregunta19']).size().reset_index(name='count')

In [ ]:
len(df_to_analyze)

In [ ]:

orden_pre = ['1-2 horas', '3-4 horas', '5-6 horas', '7 horas o más']
orden_post = ['menor a 4 horas ', 'entre 4 y 7 horas', 'entre 8 y 11 horas', 'entre 12 y 15 horas ', 'mayor a 15 horas ']

df_to_analyze['pre'] = pd.Categorical(df_to_analyze['dedicacionsemanal'], categories=orden_pre, ordered=True)
df_to_analyze['post'] = pd.Categorical(df_to_analyze['pregunta19'], categories=orden_post, ordered=True)

matriz = pd.crosstab(df_to_analyze['pre'], df_to_analyze['post'], dropna=False)

plt.figure(figsize=(12, 8))
sns.heatmap(matriz, annot=True, cmap="YlGnBu", fmt='d', cbar_kws={'label': 'Número de Estudiantes'})
#plt.title('Matriz de Transición de Dedicación Semanal (Pre vs Post)', fontsize=15)
plt.xlabel('Dedicación POST-Curso', fontsize=12)
plt.ylabel('Dedicación PRE-Curso', fontsize=12)
nombre_archivo = os.path.join(imagenes_grupos_dir, "15_grafica_matriz_dedicacion.pdf")
plt.savefig(nombre_archivo, format="pdf", bbox_inches="tight", pad_inches=0.1)
plt.show()

34 alumnos que esperaban dedicar poco (1-2h) se mantuvieron en el rango bajo (menor a 4h).

70 alumnos que esperaban un esfuerzo medio (3-4h) se mantuvieron en el rango medio (4-7h).


Sin embargo, las escalas pre y post son diferentes, por lo que mejor se realiza una normalización de los valores para verlo mejor:


**Nivel| Concepto| Rango Pre original | Rango Post original**

Bajo| Dedicación mínima| 1-2 horas| menor a 4 horas

Medio,Dedicación moderada,3-4 y 5-6 horas,entre 4 y 7 horas

Alto,Dedicación intensiva,7 horas o más,"entre 8 y 11, 12-15, >15"


In [ ]:
map_pre_normalizado = {
    '1-2 horas': 'Bajo (<4h)',
    '3-4 horas': 'Medio (4-7h)',
    '5-6 horas': 'Medio (4-7h)',
    '7 horas o más': 'Alto (>7h)'
}

map_post_normalizado = {
    'menor a 4 horas ': 'Bajo (<4h)',
    'entre 4 y 7 horas': 'Medio (4-7h)',
    'entre 8 y 11 horas': 'Alto (>7h)',
    'entre 12 y 15 horas ': 'Alto (>7h)',
    'mayor a 15 horas ': 'Alto (>7h)'
}

df_to_analyze['pre_norm'] = df_to_analyze['pre'].str.strip().map(map_pre_normalizado)
df_to_analyze['post_norm'] = df_to_analyze['post'].str.strip().map(map_post_normalizado)

orden_norm = ['Bajo (<4h)', 'Medio (4-7h)', 'Alto (>7h)']

matriz_norm = pd.crosstab(
    pd.Categorical(df_to_analyze['pre_norm'], categories=orden_norm, ordered=True),
    pd.Categorical(df_to_analyze['post_norm'], categories=orden_norm, ordered=True),
    dropna=False
)

plt.figure(figsize=(8, 6))
sns.heatmap(matriz_norm, annot=True, cmap="YlGnBu", fmt='d')
#plt.title('Matriz de Transición Normalizada')
plt.xlabel('Dedicación POST (Normalizada)')
plt.ylabel('Dedicación PRE (Normalizada)')
nombre_archivo = os.path.join(imagenes_grupos_dir, "16_grafica_matriz_normalizada.pdf")
plt.savefig(nombre_archivo, format="pdf", bbox_inches="tight", pad_inches=0.1)
plt.show()

Segun la normalización anterior, se podría decir que ningún estudiante terminó el curso dedicando menos de 4 horas a la semana.

Incluso aquellos que empezaron diciendo que dedicarían poco tiempo, terminaron moviéndose a categorías de compromiso mayor (Medio o Alto).


También, se realiza un diagrama de Barras Apiladas para poder ver el porcentaje de cambio. El eje x indica el punto de partida y los colores en la barra el porcentaje de cambio.

La barra de "1-2 horas" (La primera)
La gran mayoría de los que pensaban dedicar poco tiempo terminaron en una categoría específica (color morado).

Solo una pequeña fracción (el color amarillo arriba) logró saltar a la dedicación más alta.

In [ ]:
matriz_pct = matriz.div(matriz.sum(axis=1), axis=0) * 100

matriz_pct.plot(kind='bar', stacked=True, figsize=(12, 7), colormap='viridis')
#plt.title('¿A qué categorías migraron los estudiantes según su inicio?', fontsize=14)
plt.ylabel('Porcentaje de estudiantes (%)')
plt.xlabel('Categoría inicial (Pre-curso)')
plt.legend(title='Resultado Post-curso', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=0)
nombre_archivo = os.path.join(imagenes_grupos_dir, "17_grafica_cambio_dedicacion.pdf")
plt.savefig(nombre_archivo, format="pdf", bbox_inches="tight", pad_inches=0.1)
plt.show()

**Identificar a esos estudiantes con el mayor incremento de compromiso.**


Se intenta ver si hay estudiantes que pasaron de marcar una dedicación de 1-2 horas a más de 15 horas (188)

In [ ]:
users_incremental_to_analyze = full_file[['username', 'platform', 'dedicacionsemanal', 'pregunta19']]

In [ ]:
users_incremental_to_analyze = users_incremental_to_analyze.groupby(['username', 'platform', 'dedicacionsemanal', 'pregunta19']).size().reset_index(name='count')

In [ ]:
len(users_incremental_to_analyze)

In [ ]:
users_incremental_to_analyze.head()

In [ ]:
users_incremental_to_analyze['pre'] = users_incremental_to_analyze['dedicacionsemanal'].str.strip()
users_incremental_to_analyze['post'] = users_incremental_to_analyze['pregunta19'].str.strip()


In [ ]:

estudiantes_alto_impacto = users_incremental_to_analyze[(users_incremental_to_analyze['pre'] == '1-2 horas') &
                              (users_incremental_to_analyze['post'] == 'mayor a 15 horas')]


print(f"Se encontraron {len(estudiantes_alto_impacto)} estudiantes con salto máximo de dedicación.")


También hay estudiantes que pasaron de indicar 1 a dos horas a marcar 8 a 11

In [ ]:

estudiantes_medio_impacto = users_incremental_to_analyze[(users_incremental_to_analyze['pre'] == '1-2 horas') &
                              (users_incremental_to_analyze['post'] == 'entre 8 y 11 horas')]


print(f"Se encontraron {len(estudiantes_medio_impacto)} estudiantes con salto moderado.")


14 estudiantes que pasaron de 1-2 horas a 4 a 7 horas.

In [ ]:
estudiantes_medio_impacto = users_incremental_to_analyze[(users_incremental_to_analyze['pre'] == '1-2 horas') &
                              (users_incremental_to_analyze['post'] == 'entre 4 y 7 horas')]


print(f"Se encontraron {len(estudiantes_medio_impacto)} estudiantes con salto medio de dedicación.")


Los siguientes son los que mantuvieron su rango de horas indicado en la caracterización

In [ ]:
estudiantes_no_impacto = users_incremental_to_analyze[(users_incremental_to_analyze['pre'] == '1-2 horas') &
                              (users_incremental_to_analyze['post'] == 'menor a 4 horas')]


print(f"Se encontraron {len(estudiantes_no_impacto)} estudiantes con casi ningun salto de dedicación.")


## Comparar la dedicación semanal indicada en la caracterización con la indicada en la autoevaluación (para quienes aprobaron, desertaron y los que no aprobaron)

**Relación entre la dedicación en la autoevaluación y en la nota final**

Se trae al compendio la nota final (usando lo calculado previamente de estudiantes aprobados o no).

Se quiere saber si al dedicar más horas, la nota es mejor



In [ ]:
total_curso_aprobado_char = pd.merge(
    aprobados_unicos,
    char_file,
    how='inner',
    left_on=['username', 'platform'],
    right_on=['username', 'platform'],
    indicator=False
)

print("match:", len(total_curso_aprobado_char))

In [ ]:
total_curso_aprobado_char_eval = pd.merge(
    total_curso_aprobado_char,
    eval_file,
    how='inner',
    left_on=['username', 'platform'],
    right_on=['username', 'platform'],
    indicator=False
)

print("match:", len(total_curso_aprobado_char_eval))

In [ ]:
total_curso_aprobado_char_eval.head(17)

In [ ]:
df_to_analyze_with_note = total_curso_aprobado_char_eval[['username', 'platform', 'dedicacionsemanal', 'pregunta19', 'nota_final']]


In [ ]:
map_pre_normalizado = {
    '1-2 horas': 'Bajo (<4h)',
    '3-4 horas': 'Medio (4-7h)',
    '5-6 horas': 'Medio (4-7h)',
    '7 horas o más': 'Alto (>7h)'
}

map_post_normalizado = {
    'menor a 4 horas': 'Bajo (<4h)',
    'entre 4 y 7 horas': 'Medio (4-7h)',
    'entre 8 y 11 horas': 'Alto (>7h)',
    'entre 12 y 15 horas': 'Alto (>7h)',
    'mayor a 15 horas': 'Alto (>7h)'
}

df_to_analyze_with_note['pre_norm'] = df_to_analyze_with_note['dedicacionsemanal'].str.strip().map(map_pre_normalizado)
df_to_analyze_with_note['post_norm'] = df_to_analyze_with_note['pregunta19'].str.strip().map(map_post_normalizado)

plt.figure(figsize=(10, 6))
sns.boxplot(data=df_to_analyze_with_note, x='post_norm', y='nota_final',
            order=['Bajo (<4h)', 'Medio (4-7h)', 'Alto (>7h)'], palette='Blues')

plt.title('¿A más horas, mejor nota?', fontsize=14)
plt.xlabel('Dedicación Real (Post-Curso)')
plt.ylabel('Nota Final (0-100)')
plt.axhline(y=60, color='red', linestyle='--', label='Umbral de Aprobación')
plt.legend()
plt.show()

Ahora, se crean tres perfiles sobre los datos de quienes aprobaron:

- Grupo A (Subieron): Los que dedicaron más de lo planeado.

- Grupo B (Mantuvieron): Los que se quedaron igual.

- Grupo C (Bajaron): Los que dedicaron menos de lo planeado.

¿Los que tuvieron que subir su dedicación ("Subestimadores") terminaron con mejores notas que los que ya sabían que el curso tomaría más tiempo?

In [ ]:
df_to_analyze_with_profiles = total_curso_aprobado_char_eval[['username', 'platform', 'dedicacionsemanal', 'pregunta19', 'nota_final']]
df_to_analyze_with_profiles['pre_norm'] = df_to_analyze_with_profiles['dedicacionsemanal'].str.strip().map(map_pre_normalizado)
df_to_analyze_with_profiles['post_norm'] = df_to_analyze_with_profiles['pregunta19'].str.strip().map(map_post_normalizado)


In [ ]:
ranking = {'Bajo (<4h)': 1, 'Medio (4-7h)': 2, 'Alto (>7h)': 3}

df_to_analyze_with_profiles['pre_num'] = df_to_analyze_with_profiles['pre_norm'].str.strip().map(ranking)
df_to_analyze_with_profiles['post_num'] = df_to_analyze_with_profiles['post_norm'].str.strip().map(ranking)

nulos = df_to_analyze_with_profiles[['pre_num', 'post_num']].isna().sum()
if nulos.any():
    print("Hay valores que no se pudieron mapear:")
    print(nulos)


In [ ]:
condiciones = [
    (df_to_analyze_with_profiles['post_num'] > df_to_analyze_with_profiles['pre_num']),  # Subió
    (df_to_analyze_with_profiles['post_num'] == df_to_analyze_with_profiles['pre_num']), # Mantuvo
    (df_to_analyze_with_profiles['post_num'] < df_to_analyze_with_profiles['pre_num'])   # Bajó
]

opciones = ['Subestimador (Subió)', 'Realista (Mantuvo)', 'Optimizador (Bajó)']

df_to_analyze_with_profiles['perfil_esfuerzo'] = np.select(condiciones, opciones, default='Dato Incompleto')

print(df_to_analyze_with_profiles['perfil_esfuerzo'].value_counts())

In [ ]:
analisis_perfiles = df_to_analyze_with_profiles.groupby('perfil_esfuerzo')['nota_final'].agg(['mean', 'std', 'count']).round(2)
print(analisis_perfiles)

In [ ]:

plt.figure(figsize=(12, 7))
sns.set_style("whitegrid")

ax = sns.boxplot(data=df_to_analyze_with_profiles, x='perfil_esfuerzo', y='nota_final',
                 palette='viridis', order=['Subestimador (Subió)', 'Realista (Mantuvo)', 'Optimizador (Bajó)'])

sns.stripplot(data=df_to_analyze_with_profiles, x='perfil_esfuerzo', y='nota_final',
              color='black', alpha=0.3, jitter=True)

plt.title('¿Afecta la planeación del tiempo al éxito académico?', fontsize=15)
plt.xlabel('Perfil de Esfuerzo', fontsize=12)
plt.ylabel('Nota Final', fontsize=12)
plt.axhline(y=60, color='red', linestyle='--', label='Umbral Aprobación')
plt.legend()

plt.show()

print(df_to_analyze_with_profiles.groupby('perfil_esfuerzo')['nota_final'].mean().sort_values(ascending=False))

 El grupo 'Subestimador' presenta el rendimiento más sólido y consistente, sugiriendo que una actitud de cautela inicial se traduce en notas finales superiores. Los realistas y optimizadores son casi iguales.

 El subestimador tiene las mejores notas de los 3

**Se agrega al análisis los que reprobaron**

Son solo 8 a los que se les tiene esta información

Update: 25

In [ ]:
total_curso_reprobados_char = pd.merge(
    reprobados_unicos,
    char_file,
    how='inner',
    left_on=['username', 'platform'],
    right_on=['username', 'platform'],
    indicator=False
)

print("match:", len(total_curso_reprobados_char))

In [ ]:
total_curso_reprobado_char_eval = pd.merge(
    total_curso_reprobados_char,
    eval_file,
    how='inner',
    left_on=['username', 'platform'],
    right_on=['username', 'platform'],
    indicator=False
)

print("match:", len(total_curso_reprobado_char_eval))

In [ ]:
df_to_analyze_with_note_rep = total_curso_reprobado_char_eval[['username', 'platform', 'dedicacionsemanal', 'pregunta19', 'nota_final']]


In [ ]:
map_pre_normalizado = {
    '1-2 horas': 'Bajo (<4h)',
    '3-4 horas': 'Medio (4-7h)',
    '5-6 horas': 'Medio (4-7h)',
    '7 horas o más': 'Alto (>7h)'
}

map_post_normalizado = {
    'menor a 4 horas': 'Bajo (<4h)',
    'entre 4 y 7 horas': 'Medio (4-7h)',
    'entre 8 y 11 horas': 'Alto (>7h)',
    'entre 12 y 15 horas': 'Alto (>7h)',
    'mayor a 15 horas': 'Alto (>7h)'
}

df_to_analyze_with_note_rep['pre_norm'] = df_to_analyze_with_note_rep['dedicacionsemanal'].str.strip().map(map_pre_normalizado)
df_to_analyze_with_note_rep['post_norm'] = df_to_analyze_with_note_rep['pregunta19'].str.strip().map(map_post_normalizado)



In [ ]:
ranking = {'Bajo (<4h)': 1, 'Medio (4-7h)': 2, 'Alto (>7h)': 3}

df_to_analyze_with_note_rep['pre_num'] = df_to_analyze_with_note_rep['pre_norm'].str.strip().map(ranking)
df_to_analyze_with_note_rep['post_num'] = df_to_analyze_with_note_rep['post_norm'].str.strip().map(ranking)

nulos = df_to_analyze_with_note_rep[['pre_num', 'post_num']].isna().sum()
if nulos.any():
    print("Hay valores que no se pudieron mapear:")
    print(nulos)

In [ ]:
condiciones = [
    (df_to_analyze_with_note_rep['post_num'] > df_to_analyze_with_note_rep['pre_num']),  # Subió
    (df_to_analyze_with_note_rep['post_num'] == df_to_analyze_with_note_rep['pre_num']), # Mantuvo
    (df_to_analyze_with_note_rep['post_num'] < df_to_analyze_with_note_rep['pre_num'])   # Bajó
]

opciones = ['Subestimador (Subió)', 'Realista (Mantuvo)', 'Optimizador (Bajó)']

df_to_analyze_with_note_rep['perfil_esfuerzo'] = np.select(condiciones, opciones, default='Dato Incompleto')

print(df_to_analyze_with_note_rep['perfil_esfuerzo'].value_counts())

In [ ]:

df_to_analyze_with_profiles['estado'] = 'Aprobado'
df_to_analyze_with_note_rep['estado'] = 'Desertor'

df_grafica = pd.concat([df_to_analyze_with_profiles, df_to_analyze_with_note_rep])

ranking = {'Bajo (<4h)': 0, 'Medio (4-7h)': 1, 'Alto (>7h)': 2}
df_grafica['post_num'] = df_grafica['post_norm'].str.strip().map(ranking)

Esta es la distribución de notas para los aprobados y reprobados

In [ ]:
plt.figure(figsize=(10, 6))

sns.stripplot(data=df_grafica, x='post_num', y='nota_final', hue='estado',
              palette={'Aprobado': '#2ecc71', 'Desertor': '#e74c3c'},
              alpha=0.7, jitter=True, s=10)

plt.axhline(y=60, color='gray', linestyle='--', alpha=0.6)

plt.xticks([0, 1, 2], ['Bajo (<4h)', 'Medio (4-7h)', 'Alto (>7h)'])
plt.title('Distribución de Notas: Aprobados vs Desertores', fontsize=14)
plt.ylabel('Nota Final')
plt.xlabel('Dedicación Real al Final del Curso')
plt.legend(title='Resultado')

plt.show()

# Exportar datos caracterizacion - submission para modelos

## Datos de submissions para los tres grupos

In [ ]:
full_grupos_x_submission.columns

In [ ]:
len(full_grupos_x_submission)

In [ ]:
full_grupos_x_submission[['fecha_inicio', 'fecha_final', 'solo_fecha']].head()

In [ ]:
full_grupos_x_submission.drop(columns=['_merge'], inplace=True)

In [ ]:
## ASI SE ESTÁ CALCULADO LOS DIAS_X_ACTIVIDAD
full_grupos_x_submission['solo_fecha'] = pd.to_datetime(full_grupos_x_submission['solo_fecha'])

df_tiempos_tarea = full_grupos_x_submission.groupby(['username', 'platform', 'actividad']).agg(
    fecha_inicio=('solo_fecha', 'min'),
    fecha_fin=('solo_fecha', 'max'),
    dias_x_actividad=('solo_fecha', 'nunique')
).reset_index()

In [ ]:
len(full_grupos_x_submission)

In [ ]:
full_grupos_x_submission['solo_fecha'] = pd.to_datetime(full_grupos_x_submission['solo_fecha'])

full_grupos_x_submission['dias_x_actividad'] = full_grupos_x_submission.groupby(['username', 'platform', 'actividad'])['solo_fecha'].transform('nunique')

# Al usar el unique se está diciendo cuántos dias diferentes se
# sentó la persona a trabajar en su tarea
# y eso es lo que se cuenta como dia de actividad.
# No se está calculando un rango.

In [ ]:
full_grupos_x_submission[full_grupos_x_submission['dias_x_actividad'] > 2][['username', 'actividad', 'fecha_inicio', 'fecha_final', 'solo_fecha', 'dias_x_actividad']]

In [ ]:
full_grupos_x_submission.to_csv('/content/drive/MyDrive/estudio/UNAL/new_project/data/generated/models/grupos-x-submissions.csv', index=False) # index=False prevents writing the index as a separate column

## Datos de submissions + caracterizacion para los tres grupos

In [ ]:
full_grupos_x_submissions_x_char = pd.merge(
    full_grupos_x_submission,
    char_file,
    how='inner',
    left_on=['username', 'platform'],
    right_on=['username', 'platform'],
    indicator=False
)

print("match:", len(full_grupos_x_submissions_x_char))

In [ ]:
char_file['otra_ocupacion'].value_counts(dropna=False)

In [ ]:
len(full_grupos_x_submissions_x_char)

In [ ]:
cols_texto = full_grupos_x_submissions_x_char.select_dtypes(include=['object', 'string']).columns

for col in cols_texto:
    full_grupos_x_submissions_x_char[col] = (
        full_grupos_x_submissions_x_char[col]
        .astype(str)
        .str.replace(r'[\r\n]+', ' ', regex=True)
    )

full_grupos_x_submissions_x_char.to_csv(
    '/content/drive/MyDrive/estudio/UNAL/new_project/data/generated/models/grupos-x-submissions-x-char.csv',
    index=False,
    quoting=1 # csv.QUOTE_ALL para proteger todos los campos
)

In [ ]:
full_grupos_x_submissions_x_char.to_csv('/content/drive/MyDrive/estudio/UNAL/new_project/data/generated/models/grupos-x-submissions-x-char.csv', index=False) # index=False prevents writing the index as a separate column